# Visualize Pyschological Signal Trajectories in Human-AI Chatbot Conversation Logs

This notebook takes the per-signal, per-method scores produced by `02_llm_compute_metrics_meta`
and turns them into  visualizations. Currently the visualizations include:

1. **Per-participant trajectory grids**: every active signal's own scoring method(s), by construct; one panel per (signal, method), for a single participant.
2. **Per-participant, per-signal multi-method overlays**: all methods for one signal plotted together on a shared, percentile-normalized scale, so their trajectories can
   be visually compared.
3. **Cross-method correlation matrices**: calculated across all participants, one heatmap
   per signal, showing how much methods agree with each other on real data

Signals are grouped by **construct** throughout. Currently active: **Attachment Anxiety**
(5 signals, scored by CCR/lexicon/SBERT/Zero-shot) and **Cognitive Distortions** (3 signals, scored
by lexicon/SBERT/Zero-shot)


<br>

---
<br>

**Input Files**

- `signals_config.csv` <= same file used by `02_llm_compute_metrics_meta`
- `level1_logs/{participant}_log_level1.csv` <= used for turn text (hover
  tooltips) and to identify participants
- `metrics/{participant}_{signal_slug}.csv` <= turn-level scores per signal produced by `02_llm_compute_metrics_meta`

**Output Files**

- `{participant}_trajectories.html`: one file per participant: every active signal's
  own method(s), grouped by construct
- `{participant}_{signal_slug}_all_methods.html`: one file per (participant, signal):
  every method for that signal overlaid on one percentile-normalized chart
- `{signal_slug}_method_correlation.html`: one heatmap per signal: pairwise Spearman
  correlation between methods, pooled across all participants
- `cross_method_correlations.csv`: flat export of every signal x method-pair
  correlation, for pulling into tables/figures outside the notebook
<br>

---
<br>


**Notebook Structure**

- **Part 0. Setup**: authenticate to Box, install `plotly`/`kaleido`
- **Part 1. Paths & Config**: download inputs from Box, load `signals_config.csv`,
  group active signals by construct (this grouping drives every panel's position
  throughout), plus shared helper functions (`slugify()`, `get_score_col()`)
- **Part 2. Build Trajectory Grid**: one panel per (signal, method), rows = signals,
  columns = that signal's methods, two trajectories per panel (`user`, `AI`)
- **Part 3. Run for All Participants**: builds and saves each participant's trajectory
  grid, uploads to Box
- **Part 4. Percentile-Normalized Multi-Method Trajectories**: rescales every method's
  raw scores to a percentile rank within its own pooled distribution (methods aren't on
  comparable raw scales), then builds one overlay figure per (participant, signal)
- **Part 5. Cross-Method Correlation Matrices**: pairwise Spearman correlation between
  methods per signal, pooled across all participants' real turns
- **[DEPRECATED] Ground-Truth Evaluation on Synthetic Data**: not run in the
  current pipeline -- kept as reference/scratch code only. See that section's own header
  for why.



## Part 0: Setup

In [ ]:
!pip install plotly "kaleido==0.2.1" --quiet
import sys
!{sys.executable} -m pip install "boxsdk==3.9.2" --quiet

import re
from pathlib import Path

import numpy as np
import pandas as pd

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = "colab"

from google.colab import userdata
from boxsdk import Client, OAuth2

access_token = userdata.get('BOX_DEVELOPER_TOKEN')
auth = OAuth2(client_id=None, client_secret=None, access_token=access_token)
client = Client(auth)

me = client.user().get()
print(f"Authenticated as: {me.name} ({me.login})")

## Part 1: Paths & Config

In [ ]:
# Box folder/file IDs
box_redacted_logs_folder_id = "401874734353"
box_metrics_folder_id = "404285985307"
box_signals_config_file_id = "2360417963361"
box_signal_params_file_id = "2362064252761"
box_visualizations_folder_id = "401917297462"

root_file_path = Path("/content/logs")
level1_logs_dir = root_file_path / 'level1_logs'
metrics_dir = root_file_path / 'metrics'
vis_dir = root_file_path / 'visualizations'
signals_config_path = root_file_path / 'signals_config.csv'

for d in (level1_logs_dir, metrics_dir, vis_dir):
    d.mkdir(parents=True, exist_ok=True)

text_col = "message_content"
actor_col = "actor"
timestamp_col = "timestamp"
turn_id_col = "turn_id"
participant_col = "participant_id"
conversation_title = "conversation_title"

for item in client.folder(box_redacted_logs_folder_id).get_items():
    with open(level1_logs_dir / item.name, 'wb') as f:
        client.file(item.id).download_to(f)

for item in client.folder(box_metrics_folder_id).get_items():
    with open(metrics_dir / item.name, 'wb') as f:
        client.file(item.id).download_to(f)

with open(signals_config_path, 'wb') as f:
    client.file(box_signals_config_file_id).download_to(f)

signals_config = pd.read_csv(signals_config_path)
signals_config["active"] = signals_config["active"].astype(bool)
active_signals = signals_config[signals_config["active"]]

# Set up row order in the final grid
signals_by_construct = {}
for _, row in active_signals.iterrows():
    sigs = signals_by_construct.setdefault(row["construct"], [])
    if row["signal"] not in sigs:
        sigs.append(row["signal"])

# Explicit method list: matches signals_config.csv's  "method" column values
# for consistent column order
methods_order = ["ccr", "lexicon", "sbert", "llm_zeroshot"]

#### Helper Functions
* `slugify()` matches `02_llm_compute_metrics_meta`'s filename convention. Metrics files are named
`{participant_id}_{slugify(signal_name)}_{method}.csv` (method suffix keeps CCR/lexicon/SBERT outputs for the same signal from overwriting each other).

* `get_score_col()` picks which column to plot when a signal's metrics file has more than one (e.g. CCR and SBERT have `mean_score`/`max_score`; lexicon has just `score`).

*Note: Mean is considered "more stable" so we plot it as the default score for CCR/SBERT signals*

In [ ]:
# Helper functions

# Helper for file naming: turns "Abandonment & Loss Worry" -> "abandonment_loss_worry"
def slugify(name: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", name.lower()).strip("_")


score_col_priority = ["score", "mean_score", "max_score"] # for CCR, SBERT, return mean over max

# Get correct column for scoring metric
def get_score_col(df: pd.DataFrame) -> str:
    for col in score_col_priority:
        if col in df.columns:
            return col
    base_cols = {turn_id_col, timestamp_col, actor_col, "word_count", "hover_text"}
    remaining = [c for c in df.columns if c not in base_cols]
    if not remaining:
        raise ValueError(f"No score column found. Columns: {df.columns.tolist()}")
    return remaining[0]


## Part 2: Methods to Build Trajectory Grids

* One panel per active signal, laid out in a grid: **rows = constructs, columns = the construct's signals** (in `signals_config.csv` order).

In [ ]:
actor_colors = {"user": "#4C72B0", "AI": "#DD8452"}
hover_text_max_chars = 150 # snippet of actual conversatio excerpt

method_display_names = {"ccr": "CCR", "lexicon": "Lexicon", "sbert": "SBERT"}


def build_trajectory_grid(participant_id: str, signal_dfs: dict) -> go.Figure:
    """
    Layout: constructs are placed SIDE BY SIDE as column-blocks so total grid
    size is (max signals in any construct) rows by (sum of each construct's
    method count) columns; keeps the figure roughly square-ish.
    """
    construct_order = list(signals_by_construct.keys())
    construct_rows = {c: signals_by_construct[c] for c in construct_order}

    # Get methods that actually have data for construct
    construct_methods = {}
    for c in construct_order:
        sigs = construct_rows[c]
        construct_methods[c] = [
            m for m in ["ccr", "lexicon", "sbert", "llm_zeroshot"]
            if any((c, s, m) in signal_dfs for s in sigs)
        ]

    active_constructs = [c for c in construct_order if construct_rows[c] and construct_methods[c]]
    if not active_constructs:
        raise ValueError(f"No data to plot for {participant_id}.")

    # grid's row count is largest number of signals among constructs (e.g. 5 for Attachment Anxiety)
    n_rows = max(len(construct_rows[c]) for c in active_constructs)

    # Constructs are placed left to right
    col_offsets = {}
    total_cols = 0
    for c in active_constructs:
        col_offsets[c] = total_cols
        total_cols += len(construct_methods[c])
    n_cols = total_cols


    # Only row 1 gets method-name headers
    subplot_titles = [""] * (n_rows * n_cols)
    for c in active_constructs:
        for j, method in enumerate(construct_methods[c]):
            subplot_titles[col_offsets[c] + j] = method_display_names.get(method, method)

    fig = make_subplots(
        rows=n_rows, cols=n_cols,
        subplot_titles=subplot_titles,
        vertical_spacing=min(0.12, 0.6 / max(n_rows - 1, 1)),
        horizontal_spacing=0.05,
    )

    # Make user and AI trajectories
    legend_shown = set()
    for c in active_constructs:
        sigs = construct_rows[c]
        methods_here = construct_methods[c]
        for r, signal_name in enumerate(sigs, start=1):
            for j, method in enumerate(methods_here, start=1):
                col = col_offsets[c] + j
                key = (c, signal_name, method)
                if key not in signal_dfs:
                    fig.update_xaxes(visible=False, row=r, col=col)
                    fig.update_yaxes(visible=False, row=r, col=col)
                    continue
                df = signal_dfs[key]
                score_col = get_score_col(df)

                for actor in ["user", "AI"]:
                    subset = df[df[actor_col] == actor]
                    show_legend = actor not in legend_shown
                    fig.add_trace(
                        go.Scatter(
                            x=subset.index, y=subset[score_col],
                            mode="lines+markers",
                            name=actor, legendgroup=actor,
                            line=dict(color=actor_colors[actor], width=1.5),
                            marker=dict(size=4),
                            connectgaps=False, # NaNs show up as breaks in line
                            text=subset["hover_text"],
                            hovertemplate=(
                                "turn %{x}<br>score: %{y:.3f}<br>%{text}"
                                f"<extra>{actor}</extra>"
                            ),
                            showlegend=show_legend,
                        ),
                        row=r, col=col,
                    )
                    legend_shown.add(actor)

                # hide inactive graphs for a construct (e.g. Cognitive Distortions don't use CCR method)
                fig.update_yaxes(title_text=signal_name if j == 1 else None, row=r, col=col)
                if r == len(sigs):
                    fig.update_xaxes(title_text="turn index", row=r, col=col)

        for r in range(len(sigs) + 1, n_rows + 1):
            for j in range(1, len(methods_here) + 1):
                col = col_offsets[c] + j
                fig.update_xaxes(visible=False, row=r, col=col)
                fig.update_yaxes(visible=False, row=r, col=col)

    for c in active_constructs:
        x_start = col_offsets[c] / n_cols
        x_end = (col_offsets[c] + len(construct_methods[c])) / n_cols
        fig.add_annotation(
            xref="paper", yref="paper",
            x=(x_start + x_end) / 2, y=1.08,
            text=f"<b>{c}</b>",
            showarrow=False, xanchor="center", yanchor="bottom",
            font=dict(size=13),
        )

    fig.update_layout(
        title=f"{participant_id} — Signal Trajectories by Method",
        # fig sizes
        height=max(230 * n_rows, 300) + 60,
        width=max(320 * n_cols, 560),
        legend=dict(orientation="h", y=1.14, x=0.5, xanchor="center"),
        margin=dict(t=130),
    )
    return fig

## Part 3: Build Visualizations for Participants

For each participant: loads their level1 log and all active metrics files, builds one combined trajectory grid (`build_trajectory_grid`), and saves/uploads `{participant}_trajectories.html`. Participants with no metrics files are skipped with a note.

In [ ]:
level1_files = sorted(level1_logs_dir.glob("*_level1.csv"))
PARTICIPANTS_TO_SHOW = ["P5"]  # e.g. ["P1", "P5"] to only inline-render those; None = use size-based auto-skip for everyone

if not level1_files:
    print(f"No *_level1.csv files found in {level1_logs_dir} — run the preprocessing notebook first.")

for log_path in level1_files:
    level1_df = pd.read_csv(log_path)
    participant_id = level1_df[participant_col].iloc[0] if participant_col in level1_df.columns \
        else log_path.stem

    print(f"\n{participant_id}: {len(level1_df)} turns")

    # Show text snippet excerpt if hovering on graph
    hover_text_by_turn_id = {}
    for _, row in level1_df.iterrows():
        text = str(row[text_col]) if pd.notna(row[text_col]) else ""
        if len(text) > hover_text_max_chars:
            text = text[:hover_text_max_chars] + "…"
        hover_text_by_turn_id[row[turn_id_col]] = text

    # Build graphs for active signals
    # signal_dfs = {}
    # for _, row in active_signals.iterrows():
    #     construct, signal_name, method_name = row["construct"], row["signal"], row["method"]
    #     metrics_path = metrics_dir / f"{participant_id}_{slugify(signal_name)}_{method_name}.csv"
    #     if not metrics_path.exists():
    #         print(f"  [missing] {metrics_path.name} — skipping {signal_name} ({method_name})")
    #         continue
    #     df = pd.read_csv(metrics_path, parse_dates=[timestamp_col])
    #     df["hover_text"] = df[turn_id_col].map(hover_text_by_turn_id).fillna("")
    #     signal_dfs[(construct, signal_name, method_name)] = df

    # if not signal_dfs:
    #     print("  No metrics files found — run 02_llm_compute_metrics_meta.ipynb first.")
    #     continue

    # fig = build_trajectory_grid(participant_id, signal_dfs)
    # out_path = vis_dir / f"{participant_id}_trajectories.html"
    # fig.write_html(str(out_path))
    # print(f"  [ok] {out_path.name}")

    # Large logs create huge figures that can freeze the browser trying to
    # render inline: we write the HTML either way, but only fig.show() below a
    # size threshold. Open the HTML file directly
    # for the skipped ones; it renders fine outside the notebook
    # INLINE_RENDER_TURN_LIMIT = 3000

    # if PARTICIPANTS_TO_SHOW is not None: # only show specified participants
    #     should_show = participant_id in PARTICIPANTS_TO_SHOW
    #     skip_reason = f"not in PARTICIPANTS_TO_SHOW {PARTICIPANTS_TO_SHOW}"
    # else:
    #     should_show = len(level1_df) <= INLINE_RENDER_TURN_LIMIT
    #     skip_reason = f"{len(level1_df)} turns > {INLINE_RENDER_TURN_LIMIT}"

    # if should_show:
    #     fig.show()
    # else:
    #     print(f"  [skipped inline render -- {skip_reason}; open {out_path.name} directly instead]")

In [ ]:
# # Upload visualizations back to Box (upsert: create if new, update if it already exists)
# from boxsdk.exception import BoxAPIException

# for html_file in vis_dir.glob("*.html"):
#     try:
#         client.folder(box_visualizations_folder_id).upload(str(html_file))
#         print(f"  uploaded (new) {html_file.name}")
#     except BoxAPIException as e:
#         if e.code == 'item_name_in_use':
#             existing_id = e.context_info['conflicts']['id']
#             client.file(existing_id).update_contents(str(html_file))
#             print(f"  updated (overwrote) {html_file.name}")
#         else:
#             raise

## Part 4: Percentile-Normalized Multi-Method Trajectories

The methods are on different scales, so we normalize the raw scores to make them more a little more comparable.

We use **percentile rank normalization** where each word is assigned a percentile score based on its ranking within its subgroup as done in [Ghai, 2023, arXiv](https://arxiv.org/pdf/2306.07427).

Each visualization is for a participant x signal with user and AI trajectories for each method.

In [ ]:
from scipy.stats import rankdata

method_colors = {
    "ccr": "#4C72B0",
    "lexicon": "#DD8452",
    "sbert": "#55A868",
    "llm_zeroshot": "#C44E52",
}
actor_dash = {"user": "solid", "AI": "dash"}
actor_symbol = {"user": "circle", "AI": "diamond"}


def compute_percentile_normalized(signal_name: str, method_name: str) -> pd.DataFrame:
    """
    Loads every participant's scored turns for (signal_name, method_name) and converts raw
    scores to a 0-1 percentile rank WITHIN EACH PARTICIPANT's own turns -- a turn is ranked
    only against that same participant's other turns, not against everyone else's. Ties are
    ranked with method="min" so a tied-lowest block (e.g. lexicon's zero-inflation) lands at
    percentile 0 together, rather than spread across the middle the way average-rank would.
    """
    frames = []
    suffix = f"_{slugify(signal_name)}_{method_name}.csv"
    for metrics_path in sorted(metrics_dir.glob(f"*{suffix}")):
        df = pd.read_csv(metrics_path, parse_dates=[timestamp_col])
        score_col = get_score_col(df)
        df = df.rename(columns={score_col: "raw_score"})
        df["participant_id"] = metrics_path.name[: -len(suffix)]
        frames.append(df[[turn_id_col, timestamp_col, actor_col, "participant_id", "raw_score"]])
    if not frames:
        return pd.DataFrame()

    pooled = pd.concat(frames, ignore_index=True)
    pooled["normalized_score"] = np.nan

    for participant_id, group in pooled.groupby("participant_id"):
        valid = group["raw_score"].notna()
        n_valid = int(valid.sum())
        idx = group.index[valid]
        if n_valid > 1:
            ranks = rankdata(group.loc[valid, "raw_score"], method="min")
            pooled.loc[idx, "normalized_score"] = (ranks - 1) / (n_valid - 1)
        elif n_valid == 1:
            pooled.loc[idx, "normalized_score"] = 1.0  # nothing to rank against

    return pooled

# Precompute pooled + normalized scores once per (signal, method), reused for every participant.
normalized_lookup = {}  # (construct, signal, method) -> pooled+normalized df, all participants
for construct, sigs in signals_by_construct.items():
    for signal_name in sigs:
        sig_methods = active_signals.loc[
            (active_signals["construct"] == construct) & (active_signals["signal"] == signal_name),
            "method"
        ].tolist()
        for method_name in sig_methods:
            pooled = compute_percentile_normalized(signal_name, method_name)
            if not pooled.empty:
                normalized_lookup[(construct, signal_name, method_name)] = pooled

print(f"Normalized {len(normalized_lookup)} (signal, method) combinations across all participants.")


def build_multimethod_figure(participant_id: str, signal_name: str, method_dfs: dict) -> go.Figure:
    """
    method_dfs: {method_name: DataFrame already filtered/sorted to this participant's turns,
                 with a "hover_text" column already attached}
    One combined figure: x = turn index (this participant's turns for this signal, in time
    order), y = normalized_score (0-1 percentile rank). One line per (method, actor); legend
    grouped by method (color) with actor (solid/dash) as sub-entries within each group.
    """
    fig = go.Figure()
    methods_here = [m for m in ["ccr", "lexicon", "sbert", "llm_zeroshot"] if m in method_dfs]

    for method in methods_here:
        df = method_dfs[method].reset_index(drop=True)
        color = method_colors.get(method, "#888888")
        for actor in ["user", "AI"]:
            subset = df[df[actor_col] == actor]
            if subset.empty:
                continue
            fig.add_trace(go.Scatter(
                x=subset.index, y=subset["normalized_score"],
                mode="lines+markers",
                name=actor,
                legendgroup=method,
                legendgrouptitle_text=method_display_names.get(method, method),
                line=dict(color=color, width=1.75, dash=actor_dash[actor]),
                marker=dict(size=5, symbol=actor_symbol[actor]),
                connectgaps=False,
                text=subset["hover_text"],
                hovertemplate=(
                    f"{method_display_names.get(method, method)} \u00b7 {actor}<br>"
                    "turn %{x} \u00b7 percentile: %{y:.2f}<br>%{text}<extra></extra>"
                ),
            ))

    fig.update_layout(
        title=f"{participant_id} \u2014 {signal_name} \u2014 All Methods (percentile-normalized)",
        xaxis_title="turn index",
        yaxis_title="percentile rank within method (0 = lowest, 1 = highest)",
        yaxis=dict(range=[-0.05, 1.05]),
        legend=dict(
            orientation="v", x=1.02, y=1, xanchor="left", yanchor="top",
            groupclick="togglegroup", tracegroupgap=22,
        ),
        height=440, width=820,
        margin=dict(r=200, t=70),
    )
    return fig

In [ ]:
# ============================================================
# Build per-participant figures — WITHIN-PARTICIPANT normalization (default)
# ============================================================
# PARTICIPANTS_TO_SHOW = ["P5"]
# for log_path in sorted(level1_logs_dir.glob("*_level1.csv")):
#     level1_df_tmp = pd.read_csv(log_path)
#     participant_id = level1_df_tmp[participant_col].iloc[0] if participant_col in level1_df_tmp.columns \
#         else log_path.stem

#     hover_text_by_turn_id = {}
#     for _, row in level1_df_tmp.iterrows():
#         text = str(row[text_col]) if pd.notna(row[text_col]) else ""
#         if len(text) > hover_text_max_chars:
#             text = text[:hover_text_max_chars] + "…"
#         hover_text_by_turn_id[row[turn_id_col]] = text

#     print(f"\n{participant_id}:")
#     for construct, sigs in signals_by_construct.items():
#         for signal_name in sigs:
#             method_dfs = {}
#             for method_name in methods_order:
#                 key = (construct, signal_name, method_name)
#                 if key not in normalized_lookup:
#                     continue
#                 pooled = normalized_lookup[key]
#                 this_p = pooled[pooled["participant_id"] == participant_id].sort_values(timestamp_col).copy()
#                 if not this_p.empty:
#                     this_p["hover_text"] = this_p[turn_id_col].map(hover_text_by_turn_id).fillna("")
#                     method_dfs[method_name] = this_p

#             if not method_dfs:
#                 continue

#             fig = build_multimethod_figure(participant_id, signal_name, method_dfs)
#             out_path = vis_dir / f"{participant_id}_{slugify(signal_name)}_all_methods.html"
#             fig.write_html(str(out_path))

#             print(f"  [{out_path.name} saved — open directly, not rendered inline (too many per-signal figures to show safely)]")

In [ ]:
# from boxsdk.exception import BoxAPIException

# # Upload the percentile-normalized multi-method figures to Box (upsert: create if new, update if it already exists)
# for html_file in vis_dir.glob("*_all_methods.html"):
#     try:
#         client.folder(box_visualizations_folder_id).upload(str(html_file))
#         print(f"  uploaded (new) {html_file.name}")
#     except BoxAPIException as e:
#         if e.code == 'item_name_in_use':
#             existing_id = e.context_info['conflicts']['id']
#             client.file(existing_id).update_contents(str(html_file))
#             print(f"  updated (overwrote) {html_file.name}")
#         else:
#             raise

## Optional (Playing with Figures)

### LOWESS-smoothed Signal Trajectories

In [ ]:
from statsmodels.nonparametric.smoothers_lowess import lowess


def build_weekly_trajectory_figure(participant_id: str, signal_name: str, method_dfs: dict,
                                    lowess_frac: float = 0.4) -> go.Figure:
    """
    method_dfs: {method_name: DataFrame with timestamp_col, actor_col, normalized_score,
                 hover_text -- already filtered/sorted to one participant}

    For each (method, actor), plots THREE layers:
      - LOWESS-smoothed trend: adapts locally to the data rather than using a fixed
        window, avoiding the lag and staircase artifacts a rolling mean has. Turn-count
        weighting is approximated by duplicating each week's point (capped at 20x) before
        fitting, so a sparse week still doesn't dominate a dense one.
      - weekly points: small markers at the raw (unweighted) weekly mean
      - individual turns: hidden by default (visible="legendonly")

    lowess_frac: fraction of the data span used in each local fit (statsmodels' `frac`).
    Larger = smoother/less responsive to local wiggles; smaller = tracks noise more closely.
    """
    fig = go.Figure()
    methods_here = [m for m in ["ccr", "lexicon", "sbert", "llm_zeroshot"] if m in method_dfs]

    for method in methods_here:
        df = method_dfs[method].copy()
        df[timestamp_col] = pd.to_datetime(df[timestamp_col], format="ISO8601")
        df = df.dropna(subset=[timestamp_col])
        color = method_colors.get(method, "#888888")

        for actor in ["user", "AI"]:
            subset = df[df[actor_col] == actor].sort_values(timestamp_col)
            if subset.empty:
                continue

            subset["week"] = subset[timestamp_col].dt.to_period("W").dt.start_time

            weekly = subset.groupby("week").agg(
                mean_score=("normalized_score", "mean"),
                n_turns=("normalized_score", "count"),
            ).reset_index()
            top_idx = subset.groupby("week")["normalized_score"].idxmax()
            weekly["example"] = weekly["week"].map(
                subset.loc[top_idx].set_index("week")["hover_text"]
            )

            if weekly.empty:
                continue

            weekly["week_ordinal"] = weekly["week"].map(pd.Timestamp.toordinal)

            if len(weekly) >= 2:
                # duplicate each week's point up to n_turns times (capped) to
                # approximate turn-count weighting in the LOWESS fit
                reps = weekly["n_turns"].clip(upper=20).astype(int).values
                x_expanded = np.repeat(weekly["week_ordinal"].values, reps)
                y_expanded = np.repeat(weekly["mean_score"].values, reps)

                fitted = lowess(y_expanded, x_expanded, frac=lowess_frac, return_sorted=True)
                fitted_df = pd.DataFrame(fitted, columns=["x", "y"]).drop_duplicates(subset="x")
                smoothed_map = dict(zip(fitted_df["x"], fitted_df["y"]))
                weekly["smoothed"] = weekly["week_ordinal"].map(smoothed_map)
            else:
                # not enough points to smooth -- just show the raw weekly value
                weekly["smoothed"] = weekly["mean_score"]

            # --- primary trend line: LOWESS-smoothed, bold ---
            fig.add_trace(go.Scatter(
                x=weekly["week"], y=weekly["smoothed"],
                mode="lines",
                name=actor,
                legendgroup=method,
                legendgrouptitle_text=method_display_names.get(method, method),
                line=dict(color=color, width=3, dash=actor_dash[actor]),
                customdata=np.stack([weekly["n_turns"], weekly["mean_score"], weekly["example"]], axis=-1),
                hovertemplate=(
                    f"{method_display_names.get(method, method)} \u00b7 {actor}<br>"
                    "week of %{x|%b %d, %Y}<br>"
                    "LOWESS trend: %{y:.2f} (raw week: %{customdata[1]:.2f}, n=%{customdata[0]})<br>"
                    "highest-scoring turn: %{customdata[2]}<extra></extra>"
                ),
            ))

            # --- weekly points: small, subordinate to the trend line, still visible ---
            fig.add_trace(go.Scatter(
                x=weekly["week"], y=weekly["mean_score"],
                mode="markers",
                legendgroup=method,
                showlegend=False,
                marker=dict(size=5, symbol=actor_symbol[actor], color=color, opacity=0.55),
                hoverinfo="skip",
            ))

            # --- individual turns: OFF by default, toggle-able from the legend ---
            fig.add_trace(go.Scatter(
                x=subset[timestamp_col], y=subset["normalized_score"],
                mode="markers",
                legendgroup=method,
                name=f"{actor} (all turns)",
                showlegend=True,
                visible="legendonly",
                marker=dict(size=4, symbol=actor_symbol[actor], color=color, opacity=0.35, line=dict(width=0)),
                text=subset["hover_text"],
                hovertemplate=(
                    f"{method_display_names.get(method, method)} \u00b7 {actor} turn<br>"
                    "%{x|%b %d, %Y}<br>percentile: %{y:.2f}<br>%{text}<extra></extra>"
                ),
            ))

    fig.update_layout(
        title=f"{participant_id} \u2014 {signal_name} \u2014 Weekly Trajectory (LOWESS-smoothed)",
        xaxis_title="week",
        yaxis_title="percentile rank within method (0 = lowest, 1 = highest)",
        yaxis=dict(range=[-0.05, 1.05]),
        legend=dict(
            orientation="v", x=1.02, y=1, xanchor="left", yanchor="top",
            groupclick="togglegroup", tracegroupgap=22,
        ),
        height=440, width=860,
        margin=dict(r=200, t=70),
    )
    return fig

PARTICIPANTS_TO_SHOW = ["P5"]
for log_path in sorted(level1_logs_dir.glob("*_level1.csv")):
    level1_df_tmp = pd.read_csv(log_path)
    participant_id = level1_df_tmp[participant_col].iloc[0] if participant_col in level1_df_tmp.columns \
        else log_path.stem

    hover_text_by_turn_id = {}
    for _, row in level1_df_tmp.iterrows():
        text = str(row[text_col]) if pd.notna(row[text_col]) else ""
        if len(text) > hover_text_max_chars:
            text = text[:hover_text_max_chars] + "\u2026"
        hover_text_by_turn_id[row[turn_id_col]] = text

    print(f"\n{participant_id}:")
    for construct, sigs in signals_by_construct.items():
        for signal_name in sigs:
            method_dfs = {}
            for method_name in ["ccr", "lexicon", "sbert", "llm_zeroshot"]:
                key = (construct, signal_name, method_name)
                if key not in normalized_lookup:
                    continue
                pooled = normalized_lookup[key]
                this_p = pooled[pooled["participant_id"] == participant_id].sort_values(timestamp_col).copy()
                if not this_p.empty:
                    this_p["hover_text"] = this_p[turn_id_col].map(hover_text_by_turn_id).fillna("")
                    method_dfs[method_name] = this_p

            if not method_dfs:
                continue

            fig = build_weekly_trajectory_figure(participant_id, signal_name, method_dfs)
            out_path = vis_dir / f"{participant_id}_{slugify(signal_name)}_weekly_trajectory.html"
            fig.write_html(str(out_path))
            print(f"  [ok] {out_path.name}  (methods: {list(method_dfs.keys())})")

            # if PARTICIPANTS_TO_SHOW is None or participant_id in PARTICIPANTS_TO_SHOW:
            #     fig.show()
            # else:
            #     print(f"  [skipped inline render for {participant_id} — not in PARTICIPANTS_TO_SHOW]")

In [ ]:
# from boxsdk.exception import BoxAPIException

# # Upload the weekly trajectory figures to Box (upsert: create if new, update if it already exists)
# for html_file in vis_dir.glob("*_weekly_trajectory.html"):
#     try:
#         client.folder(box_visualizations_folder_id).upload(str(html_file))
#         print(f"  uploaded (new) {html_file.name}")
#     except BoxAPIException as e:
#         if e.code == 'item_name_in_use':
#             existing_id = e.context_info['conflicts']['id']
#             client.file(existing_id).update_contents(str(html_file))
#             print(f"  updated (overwrote) {html_file.name}")
#         else:
#             raise

In [ ]:
# from plotly.subplots import make_subplots
# from statsmodels.nonparametric.smoothers_lowess import lowess

# def build_participant_weekly_grid(participant_id: str, actor: str = "user",
#                                    lowess_frac: float = 0.15) -> go.Figure:
#     """
#     One page per participant: every active signal gets its own row, all methods
#     overlaid within that row (weekly mean + LOWESS trend). Reuses `normalized_lookup`
#     from Part 4 -- no rescoring, just re-plots what's already been computed.
#     """
#     row_signals = [
#         (construct, signal_name)
#         for construct, sigs in signals_by_construct.items()
#         for signal_name in sigs
#     ]
#     n_rows = len(row_signals)
#     if n_rows == 0:
#         raise ValueError("No active signals found in signals_by_construct.")

#     fig = make_subplots(
#         rows=n_rows, cols=1,
#         subplot_titles=[f"{construct} \u2014 {signal_name}" for construct, signal_name in row_signals],
#         vertical_spacing=min(0.05, 0.9 / max(n_rows - 1, 1)),
#     )

#     legend_shown = set()

#     for r, (construct, signal_name) in enumerate(row_signals, start=1):
#         for method_name in methods_order:
#             key = (construct, signal_name, method_name)
#             if key not in normalized_lookup:
#                 continue
#             pooled = normalized_lookup[key]
#             this_p = pooled[pooled["participant_id"] == participant_id].sort_values(timestamp_col).copy()
#             this_p = this_p[this_p[actor_col] == actor]
#             if this_p.empty:
#                 continue

#             this_p[timestamp_col] = pd.to_datetime(this_p[timestamp_col], format="ISO8601")
#             this_p["week"] = this_p[timestamp_col].dt.to_period("W").dt.start_time
#             weekly = this_p.groupby("week").agg(
#                 mean_score=("normalized_score", "mean"),
#                 n_turns=("normalized_score", "count"),
#             ).reset_index()

#             color = method_colors.get(method_name, "#888888")
#             label = method_display_names.get(method_name, method_name)
#             show_legend = method_name not in legend_shown

#             if len(weekly) >= 2:
#                 weekly["week_ordinal"] = weekly["week"].map(pd.Timestamp.toordinal)
#                 reps = weekly["n_turns"].clip(upper=20).astype(int).values
#                 x_expanded = np.repeat(weekly["week_ordinal"].values, reps)
#                 y_expanded = np.repeat(weekly["mean_score"].values, reps)
#                 fitted = lowess(y_expanded, x_expanded, frac=lowess_frac, return_sorted=True)
#                 fitted_df = pd.DataFrame(fitted, columns=["x", "y"]).drop_duplicates(subset="x")
#                 smoothed_map = dict(zip(fitted_df["x"], fitted_df["y"]))
#                 weekly["smoothed"] = weekly["week_ordinal"].map(smoothed_map)
#             else:
#                 weekly["smoothed"] = weekly["mean_score"]

#             fig.add_trace(go.Scatter(
#                 x=weekly["week"], y=weekly["smoothed"], mode="lines",
#                 name=label, legendgroup=method_name, showlegend=show_legend,
#                 line=dict(color=color, width=2),
#             ), row=r, col=1)
#             fig.add_trace(go.Scatter(
#                 x=weekly["week"], y=weekly["mean_score"], mode="markers",
#                 legendgroup=method_name, showlegend=False,
#                 marker=dict(size=4, color=color, opacity=0.5),
#             ), row=r, col=1)

#             legend_shown.add(method_name)

#         fig.update_yaxes(range=[-0.05, 1.05], row=r, col=1)

#     fig.update_layout(
#       title=f"{participant_id} — All Active Signals, Weekly Trajectories ({actor} turns)",
#       height=max(280 * n_rows, 400),   # bumped from 220 -> 280 per row
#       width=760,                        # narrower, so tall/narrow reads correctly
#       legend=dict(orientation="h", y=1.02, x=0.5, xanchor="center"),
#       margin=dict(t=100),
#     )
#     return fig


# def save_participant_weekly_grid(participant_id: str, actor: str = "user"):
#     fig = build_participant_weekly_grid(participant_id, actor=actor)
#     out_path = vis_dir / f"{participant_id}_all_signals_weekly_grid.html"
#     fig.write_html(str(out_path))
#     # print(f"[ok] {out_path.name}")
#     # fig.show()
#     return fig



### Hero Figure Option

Reuses `build_weekly_trajectory_figure` and `normalized_lookup` from above --
no new plotting logic, just picks one participant/signal and layers on a
handful of example turns sampled across the *entire* log (not just the crisis
window), so the poster panel shows a baseline-to-crisis story arc.

**Set `HERO_PARTICIPANT` / `HERO_CONSTRUCT` / `HERO_SIGNAL` to match specific
`signals_config.csv` values exactly** (case-sensitive) before running.

In [ ]:
HERO_PARTICIPANT = "P5"
HERO_CONSTRUCT = "Attachment Anxiety"   # match signals_config.csv 'construct' value
HERO_SIGNAL = "Fear_of_Being_Known"    # match signals_config.csv 'signal' value

# Rebuild the hover-text lookup for just the hero participant
hero_level1_path = level1_logs_dir / f"{HERO_PARTICIPANT}_log_level1.csv"
hero_level1_df = pd.read_csv(hero_level1_path)

hero_hover_text_by_turn_id = {}
for _, row in hero_level1_df.iterrows():
    text = str(row[text_col]) if pd.notna(row[text_col]) else ""
    if len(text) > hover_text_max_chars:
        text = text[:hover_text_max_chars] + "…"
    hero_hover_text_by_turn_id[row[turn_id_col]] = text

hero_method_dfs = {}
for method_name in methods_order:
    key = (HERO_CONSTRUCT, HERO_SIGNAL, method_name)
    if key not in normalized_lookup:
        print(f"  [skip] no normalized data for {key}")
        continue
    pooled = normalized_lookup[key]
    this_p = pooled[pooled["participant_id"] == HERO_PARTICIPANT].sort_values(timestamp_col).copy()
    if this_p.empty:
        continue
    this_p["hover_text"] = this_p[turn_id_col].map(hero_hover_text_by_turn_id).fillna("")
    hero_method_dfs[method_name] = this_p

# Same function already defined above (cell with the LOWESS trend line) -- reused as-is.
hero_fig = build_weekly_trajectory_figure(
    HERO_PARTICIPANT, HERO_SIGNAL, hero_method_dfs,
    lowess_frac=0.08,   # ~8% of the span instead of 40% -- keeps local spikes visible
)

hero_fig.show()

### Excerpt Validation

Metrics Checker: enter excerpts and check the scores for each of the methods (except lexicon)

Check some chatlog excerpts against its percentile rank within
CCR, SBERT, and zero-shot LLM scoring (lexicon excluded).

*Note: Percentile is arbitrary, 90th for now*

In [ ]:
# ============================================================
# Turn Lookup: table of raw AND normalized (within-participant) scores
# for chosen turn(s), across all active signals and methods.
# Requires normalized_lookup (cell 13) to have already run.
# ============================================================

def get_turn_scores_both(participant_id: str, turn_ids, signals=None) -> pd.DataFrame:
    """
    Look up both raw and normalized scores for one or more turns, across all
    active signals and methods.

    Returns (long_df, context_df):
      long_df    - one row per (turn_id, construct, signal, method), with
                   both "raw_score" and "normalized_score" columns
      context_df - actor/timestamp/message_content for the requested turns,
                   indexed by turn_id
    """
    if not isinstance(turn_ids, (list, tuple, set)):
        turn_ids = [turn_ids]
    turn_ids = list(turn_ids)

    level1_path = level1_logs_dir / f"{participant_id}_log_level1.csv"
    if not level1_path.exists():
        raise FileNotFoundError(f"No level1 log found for {participant_id} at {level1_path}")
    level1_df = pd.read_csv(level1_path)
    context_df = level1_df[level1_df[turn_id_col].isin(turn_ids)].set_index(turn_id_col)

    rows_to_check = active_signals if signals is None \
        else active_signals[active_signals["signal"].isin(signals)]

    records = []
    for _, row in rows_to_check.iterrows():
        construct, signal_name, method_name = row["construct"], row["signal"], row["method"]

        # --- raw score: straight from the metrics CSV ---
        metrics_path = metrics_dir / f"{participant_id}_{slugify(signal_name)}_{method_name}.csv"
        raw_by_tid = {}
        if metrics_path.exists():
            raw_df = pd.read_csv(metrics_path)
            score_col = get_score_col(raw_df)
            raw_df = raw_df.set_index(turn_id_col)
            for tid in turn_ids:
                if tid in raw_df.index:
                    val = raw_df.loc[tid, score_col]
                    raw_by_tid[tid] = val.iloc[0] if isinstance(val, pd.Series) else val

        # --- normalized score: from normalized_lookup (within-participant percentile) ---
        norm_by_tid = {}
        key = (construct, signal_name, method_name)
        if key in normalized_lookup:
            norm_df = normalized_lookup[key]
            norm_df = norm_df[norm_df["participant_id"] == participant_id].set_index(turn_id_col)
            for tid in turn_ids:
                if tid in norm_df.index:
                    val = norm_df.loc[tid, "normalized_score"]
                    norm_by_tid[tid] = val.iloc[0] if isinstance(val, pd.Series) else val

        for tid in turn_ids:
            records.append({
                "turn_id": tid, "construct": construct,
                "signal": signal_name, "method": method_name,
                "raw_score": raw_by_tid.get(tid, np.nan),
                "normalized_score": norm_by_tid.get(tid, np.nan),
            })

    return pd.DataFrame(records), context_df


def display_turn_table_both(participant_id: str, turn_ids, signals=None):
    """Pretty-print a wide table (method x [raw, normalized]) for each requested
    turn, with the message text/actor/timestamp shown as context above it."""
    if not isinstance(turn_ids, (list, tuple, set)):
        turn_ids = [turn_ids]

    long_df, context_df = get_turn_scores_both(participant_id, turn_ids, signals=signals)

    for tid in turn_ids:
        print(f"\n{'='*70}")
        if tid in context_df.index:
            ctx = context_df.loc[tid]
            text = str(ctx[text_col]) if pd.notna(ctx[text_col]) else ""
            if len(text) > 200:
                text = text[:200] + "…"
            print(f"Turn {tid}  |  {participant_id}  |  actor={ctx[actor_col]}  |  {ctx[timestamp_col]}")
            print(f'  "{text}"')
        else:
            print(f"Turn {tid}  |  {participant_id}  (not found in level1 log)")
        print(f"{'='*70}")

        sub = long_df[long_df["turn_id"] == tid]
        if sub.empty:
            print("  No metrics found for this turn.")
            continue

        wide = sub.pivot_table(
            index=["construct", "signal"], columns="method",
            values=["raw_score", "normalized_score"], aggfunc="first",
        )
        # reorder so each method's raw+normalized sit next to each other, methods in methods_order
        present_methods = [m for m in methods_order if m in wide.columns.get_level_values(1)]
        wide = wide.reorder_levels([1, 0], axis=1)
        wide = wide.reindex(columns=pd.MultiIndex.from_product(
            [present_methods, ["raw_score", "normalized_score"]]
        ))
        wide.columns = wide.columns.set_levels(
            [method_display_names.get(m, m) for m in wide.columns.levels[0]], level=0
        )
        display(wide.round(3))

    return long_df


# ---- Edit these and re-run ----
LOOKUP_PARTICIPANT = "P5"
LOOKUP_TURN_IDS = ["P5_6868a78b-b390-8013-8d0c-7c4ac4497e3c_9"]   # <- turn_id(s) you want to inspect

_ = display_turn_table_both(LOOKUP_PARTICIPANT, LOOKUP_TURN_IDS)

In [ ]:
# HERO FIGURE: full trajectory, user vs. AI

legend_order = ["ccr", "sbert", "llm_zeroshot"]  # lexicon removed

hero_solo_fig = go.Figure()

plotted_date_min, plotted_date_max = None, None

for method_name in legend_order:
    df = hero_method_dfs.get(method_name)
    if df is None:
        continue
    color = method_colors.get(method_name, "#888888")

    for actor in ["user", "AI"]:
        actor_df = df[df[actor_col] == actor].copy()
        if actor_df.empty:
            continue

        actor_df[timestamp_col] = pd.to_datetime(actor_df[timestamp_col], errors="coerce", utc=True).dt.tz_localize(None)
        n_failed = actor_df[timestamp_col].isna().sum()
        if n_failed > 0:
            print(f"Warning: {n_failed} timestamps failed to parse and will be dropped")
            actor_df = actor_df.dropna(subset=[timestamp_col])

        weekly_stats = (
            actor_df.set_index(timestamp_col)["normalized_score"]
            .resample("W")
            .agg(["mean", "count"])
        )
        weekly_stats = weekly_stats.dropna(subset=["mean"])

        # Drop weeks where the 4-week window couldn't be reasonably filled
        MIN_WEEKS_FOR_STABLE_SMOOTH = 2
        weekly_stats = weekly_stats.iloc[MIN_WEEKS_FOR_STABLE_SMOOTH:-MIN_WEEKS_FOR_STABLE_SMOOTH] if len(weekly_stats) > 2 * MIN_WEEKS_FOR_STABLE_SMOOTH else weekly_stats

        if weekly_stats.empty:
            continue

        plotted_date_min = weekly_stats.index.min() if plotted_date_min is None else min(plotted_date_min, weekly_stats.index.min())
        plotted_date_max = weekly_stats.index.max() if plotted_date_max is None else max(plotted_date_max, weekly_stats.index.max())

        num = (weekly_stats["mean"] * weekly_stats["count"]).rolling(
            4, center=True, min_periods=1, win_type="gaussian"
        ).mean(std=1.0)
        denom = weekly_stats["count"].rolling(
            4, center=True, min_periods=1, win_type="gaussian"
        ).mean(std=1.0)
        weekly_stats["weighted"] = (num / denom).clip(0, 1)

        hero_solo_fig.add_trace(go.Scatter(
            x=weekly_stats.index, y=weekly_stats["weighted"], mode="lines+markers",
            name=f"{method_display_names.get(method_name, method_name)} · {actor}",
            legendgroup=method_name,
            legendgrouptitle_text=method_display_names.get(method_name, method_name),
            line=dict(color=color, width=5, dash=actor_dash[actor], shape="spline", smoothing=1.0),
            marker=dict(size=10, symbol=actor_symbol[actor]),
            connectgaps=False,
        ))

if plotted_date_min is None:
    raise ValueError("No data survived the trim -- MIN_WEEKS_FOR_STABLE_SMOOTH may be too high for this signal.")

# footer note -- annotation-related line removed since there are no annotated excerpts anymore
hero_solo_fig.add_annotation(
    text=(
        "Note: Lexicon scoring omitted above; remained flat throughout.<br>"
        "Scores are weekly-aggregated and smoothed (4-week Gaussian rolling average)."
    ),
    xref="paper", yref="paper", x=0.0, y=-0.14,
    showarrow=False, font=dict(size=22, color="#333333"),
    align="left", xanchor="left", yanchor="top",
)

# --- x-axis range: derived from what's actually PLOTTED (post-trim) ---
date_min = plotted_date_min - pd.Timedelta(days=14)
date_max = plotted_date_max + pd.Timedelta(days=14)
print(f"Plotted range: {date_min.date()} to {date_max.date()}")

hero_solo_fig.update_xaxes(
    range=[date_min, date_max],
    title_text="Time",
    title_font=dict(size=40),
    tickfont=dict(size=28),
    linewidth=2, gridwidth=1,
)
hero_solo_fig.update_yaxes(
    range=[0, 1.0],
    title_text="Percentile Rank Within Method",
    title_font=dict(size=40),
    tickfont=dict(size=28),
    linewidth=2, gridwidth=1,
)
hero_solo_fig.update_layout(
    title=dict(
        text=f"{HERO_PARTICIPANT} — {HERO_SIGNAL} — Full Trajectory (User vs. AI)",
        font=dict(size=44),
        y=0.99,
    ),
    height=1600, width=3400,
    margin=dict(t=420, l=170, r=340, b=280),
    legend=dict(
        orientation="v",
        x=1.03, y=0.55, xanchor="left", yanchor="middle",
        groupclick="togglegroup", tracegroupgap=28,
        font=dict(size=24), grouptitlefont=dict(size=26),
    ),
    plot_bgcolor="white", paper_bgcolor="white",
)

hero_solo_fig.show()

# --- Export: HTML (interactive reference) + high-res PNG (for Slides/print) ---
hero_solo_fig.write_html(f"hero_figure_{HERO_PARTICIPANT}_{slugify(HERO_SIGNAL)}.html")
png_path = f"hero_figure_{HERO_PARTICIPANT}_{slugify(HERO_SIGNAL)}.png"
hero_solo_fig.write_image(png_path, width=3400, height=1600, scale=5)

from google.colab import files
files.download(f"hero_figure_{HERO_PARTICIPANT}_{slugify(HERO_SIGNAL)}.html")
files.download(png_path)

In [ ]:
# ============================================================
# List all turns where CCR, SBERT, AND zero-shot all agree
# above a given percentile threshold (lexicon excluded)
# ============================================================

import numpy as np
from scipy.stats import rankdata

THRESHOLD = 0.9        # <- change this to adjust the bar
ACTOR_FILTER = None   # <- "user", "AI", or None to include both


def compute_pooled_percentiles(csv_path, score_col):
    df = pd.read_csv(csv_path)
    valid = df[score_col].notna()
    n = valid.sum()
    pct = pd.Series(np.nan, index=df.index)
    if n > 1:
        ranks = rankdata(df.loc[valid, score_col], method="min")
        pct.loc[valid] = (ranks - 1) / (n - 1)
    df["_pct"] = pct
    return df.set_index(turn_id_col)["_pct"].to_dict()


method_files = {
    "ccr": (metrics_dir / f"{HERO_PARTICIPANT}_{slugify(HERO_SIGNAL)}_ccr.csv", "mean_score"),
    "sbert": (metrics_dir / f"{HERO_PARTICIPANT}_{slugify(HERO_SIGNAL)}_sbert.csv", "mean_score"),
    "llm_zeroshot": (metrics_dir / f"{HERO_PARTICIPANT}_{slugify(HERO_SIGNAL)}_llm_zeroshot.csv", "score"),
}
pct_lookup = {m: compute_pooled_percentiles(path, col) for m, (path, col) in method_files.items()}

hero_level1_df["ccr_pctl"] = hero_level1_df[turn_id_col].map(pct_lookup["ccr"])
hero_level1_df["sbert_pctl"] = hero_level1_df[turn_id_col].map(pct_lookup["sbert"])
hero_level1_df["zeroshot_pctl"] = hero_level1_df[turn_id_col].map(pct_lookup["llm_zeroshot"])

mask_all3 = (
    (hero_level1_df["ccr_pctl"] >= THRESHOLD) &
    (hero_level1_df["sbert_pctl"] >= THRESHOLD) &
    (hero_level1_df["zeroshot_pctl"] >= THRESHOLD)
)
if ACTOR_FILTER is not None:
    mask_all3 = mask_all3 & (hero_level1_df[actor_col] == ACTOR_FILTER)

agree = hero_level1_df[mask_all3].copy()
agree["row"] = agree.index + 2   # spreadsheet row = 0-indexed position + 2 (header row)

print_cols = ["row", "timestamp", actor_col, "message_content", "ccr_pctl", "sbert_pctl", "zeroshot_pctl"]
display_df = agree[print_cols].copy()
display_df["message_content"] = display_df["message_content"].astype(str).str.slice(0, 80)
display_df[["ccr_pctl", "sbert_pctl", "zeroshot_pctl"]] = display_df[["ccr_pctl", "sbert_pctl", "zeroshot_pctl"]].round(3)

actor_label = ACTOR_FILTER if ACTOR_FILTER is not None else "user + AI"
print(f"{len(agree)} {actor_label} turns at or above the {THRESHOLD:.0%} percentile across CCR, SBERT, and zero-shot.")
display(display_df)

## Part 6: Participant Manual Validation: F1 Against Manual Codes

Validates method scores against hand-coded scores on a participant's real log, using an
0/1/2 manual coding scale (0 = absent, 1 = implicit/weak signal, 2 = explicit; collapsed to binary
present(1/2) and absent(0) for F1). **Lexicon is excluded** from this validation. Covers every signal currently marked `active=True` in `signals_config.csv`.

**Workflow:**
1. Run 6.1 to build a fill-in template covering every turn in a participant's log.
2. Fill in the `manual_score` column (0/1/2) per signal per the codebook, save as
   `PX_manual_coding_FILLED.csv` in the same folder.
3. Run 6.2 to merge your codes with the metrics files and compute F1 per (signal, method), with macro-averages.
4. Run 6.3 for a heatmap

*Note: Currently use P5*

In [ ]:
# ============================================================
# Part 6.1: Build manual coding template: ALL of PX's turns
# ============================================================

F1_METHODS = ["ccr", "sbert", "llm_zeroshot"]   # lexicon excluded
VALIDATION_PARTICIPANT = "P5"   # change this to validate a different participant

manual_coding_dir = root_file_path / "manual_coding"
manual_coding_dir.mkdir(parents=True, exist_ok=True)


def load_px_metrics_long():
    """
    Reads every {VALIDATION_PARTICIPANT}_{slugify(signal)}_{method}.csv in metrics_dir that
    exists, and stacks them into one long dataframe: turn_id, signal, method, score.
    Covers every active signal in signals_by_construct. Missing (signal, method) combos
    are absent (e.g. CCR will always be missing for signals with
    no anchor_items (Cognitive Distortions)).
    """
    rows = []
    for construct, sigs in signals_by_construct.items():
        for signal_name in sigs:
            for method in F1_METHODS:
                path = metrics_dir / f"{VALIDATION_PARTICIPANT}_{slugify(signal_name)}_{method}.csv"
                if not path.exists():
                    continue
                df = pd.read_csv(path)
                score_col = get_score_col(df)
                sub = df[[turn_id_col, score_col]].rename(columns={score_col: "score"})
                sub["signal"] = signal_name
                sub["method"] = method
                rows.append(sub)
    if not rows:
        raise FileNotFoundError(
            f"No metrics files found for {VALIDATION_PARTICIPANT} in {metrics_dir}. "
            f"Run notebook 02 for PX first (check signal_methods is active for the methods you want)."
        )
    return pd.concat(rows, ignore_index=True)


def build_manual_coding_template():
    """
    One row per turn_id -- EVERY turn in PX's level1 log, not a sample --
    one column per active signal, pre-filled with an excerpt preview of the
    actual (redacted) message so you're not cross-referencing the raw log
    while coding.
    """
    level1_path = level1_logs_dir / f"{VALIDATION_PARTICIPANT}_log_level1.csv"
    if not level1_path.exists():
        raise FileNotFoundError(
            f"{level1_path} not found -- check PX's redacted log made it into level1_logs_dir."
        )
    turns_df = pd.read_csv(level1_path)

    template = turns_df[[turn_id_col, conversation_title, actor_col, timestamp_col, text_col]].copy()
    template = template.rename(columns={text_col: "message_text"}).sort_values(timestamp_col)
    template.insert(0, "participant_id", VALIDATION_PARTICIPANT)

    all_signals = [s for sigs in signals_by_construct.values() for s in sigs]
    for signal in all_signals:
        template[signal] = ""   # fill with 0 / 1 / 2

    template["annotator_id"] = ""
    template["notes"] = ""

    out_path = manual_coding_dir / f"{VALIDATION_PARTICIPANT}_manual_coding_TEMPLATE.csv"
    template.to_csv(out_path, index=False)
    print(f"Template written to {out_path} "
          f"({len(template)} turns x {len(all_signals)} signals).")
    print("Fill in 0 / 1 / 2 per signal per the codebook, then save as "
          f"{VALIDATION_PARTICIPANT}_manual_coding_FILLED.csv in the same folder ({manual_coding_dir}).")
    return template


px_metrics_long = load_px_metrics_long()
manual_template = build_manual_coding_template()

### Fill in the template now

Open `PX_manual_coding_TEMPLATE.csv`, code each signal column 0/1/2/NA per the codebook,
save it as `PX_manual_coding_FILLED.csv` in the same `manual_coding/` folder, then run the
cell below.

In [ ]:
# ============================================================
# Part 6.2: F1 per (signal, method), threshold-swept, macro-averaged
# ============================================================
from sklearn.metrics import precision_recall_curve


def load_manual_coding(path=None):
    path = path or (manual_coding_dir / f"{VALIDATION_PARTICIPANT}_manual_coding_FILLED.csv")
    if not path.exists():
        raise FileNotFoundError(f"{path} not found -- fill in the template and save it there first.")
    wide = pd.read_csv(path)

    all_signals = [s for sigs in signals_by_construct.values() for s in sigs]
    long = wide.melt(
        id_vars=[turn_id_col],
        value_vars=[s for s in all_signals if s in wide.columns],
        var_name="signal", value_name="manual_score",
    )
    # Blank cells = 0 (not scored above absent). N/A isn't part of the
    # coding scale, so nothing gets dropped here -- every turn contributes.
    # pd.to_numeric handles blanks, NaN, and "0.0"-style float strings alike,
    # which plain int()/str() conversion can't (int("0.0") raises ValueError).
    long["manual_score"] = pd.to_numeric(long["manual_score"], errors="coerce").fillna(0)
    long["manual_score"] = long["manual_score"].astype(int)
    long["manual_present"] = (long["manual_score"] >= 1).astype(int)
    return long


def best_f1_for_group(y_true, y_score):
    if y_true.sum() == 0 or y_true.sum() == len(y_true):
        return {"precision": np.nan, "recall": np.nan, "f1": np.nan, "threshold": np.nan,
                "n_positive": int(y_true.sum()), "n_total": len(y_true)}
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_score)
    f1s = 2 * (precisions * recalls) / (precisions + recalls + 1e-12)
    best_idx = np.nanargmax(f1s[:-1])
    return {"precision": precisions[best_idx], "recall": recalls[best_idx], "f1": f1s[best_idx],
            "threshold": thresholds[best_idx], "n_positive": int(y_true.sum()), "n_total": len(y_true)}


manual_long = load_manual_coding()
merged = px_metrics_long.merge(
    manual_long[[turn_id_col, "signal", "manual_present"]],
    on=[turn_id_col, "signal"], how="inner",
)

f1_results = []
for signal in sorted(merged["signal"].unique()):
    for method in F1_METHODS:
        subset = merged[(merged["signal"] == signal) & (merged["method"] == method)]
        subset = subset.dropna(subset=["score"])
        if subset.empty:
            continue
        res = best_f1_for_group(subset["manual_present"].values, subset["score"].values)
        res["n_dropped_missing_score"] = (
            merged[(merged["signal"] == signal) & (merged["method"] == method)]["score"].isna().sum()
        )
        res.update({"signal": signal, "method": method})
        f1_results.append(res)

f1_df = pd.DataFrame(f1_results)[
    ["signal", "method", "n_total", "n_positive", "threshold", "precision", "recall", "f1"]
]
macro_f1 = f1_df.groupby("method")["f1"].mean().reset_index().rename(columns={"f1": "macro_f1"})

print("=== Per-signal F1 (best threshold, in-sample) ===")
print(f1_df.round(3).to_string(index=False))
print("\n=== Macro-averaged F1 across signals, per method ===")
print(macro_f1.round(3).to_string(index=False))
print("\nReminder: thresholds above are fit TO this P5 sample -- report as an in-sample "
      "best-case, not a generalizable operating point. n_positive below ~10 should be "
      "flagged as low-confidence in any write-up.")

f1_out_path = vis_dir / f"{VALIDATION_PARTICIPANT}_f1_validation_summary_v2.csv"
f1_df.to_csv(f1_out_path, index=False)
print(f"\n[ok] {f1_out_path.name}")

In [ ]:
# TRUST_THRESHOLD = 10  # n_positive floor for inclusion

# trustworthy = f1_df[f1_df["n_positive"] >= TRUST_THRESHOLD]
# macro_f1_trustworthy = (trustworthy.groupby("method")["f1"].mean()
#                          .reset_index().rename(columns={"f1": "macro_f1_trustworthy"}))

# print(f"Signals meeting n_positive >= {TRUST_THRESHOLD}:", sorted(trustworthy["signal"].unique()))
# print(macro_f1_trustworthy.round(3).to_string(index=False))

In [ ]:
# common_signals = (trustworthy.groupby("signal")["method"].nunique())
# common_signals = common_signals[common_signals == len(F1_METHODS)].index.tolist()
# print("Signals with all 3 methods:", common_signals)

# fair = trustworthy[trustworthy["signal"].isin(common_signals)]
# macro_f1_fair = fair.groupby("method")["f1"].mean().reset_index().rename(columns={"f1": "macro_f1_fair"})
# print(macro_f1_fair.round(3).to_string(index=False))

In [ ]:
# ============================================================
# All turns flagged "positive" by each method,
# at that (signal, method)'s best-F1 threshold
# ============================================================

# level1_path = level1_logs_dir / f"{VALIDATION_PARTICIPANT}_log_level1.csv"
# level1_df_full = pd.read_csv(level1_path).set_index(turn_id_col)

# detected_rows = []
# for _, row in f1_df.dropna(subset=["threshold"]).iterrows():
#     signal, method, threshold = row["signal"], row["method"], row["threshold"]

#     subset = merged[(merged["signal"] == signal) & (merged["method"] == method)]
#     positives = subset[subset["score"] >= threshold].copy()
#     if positives.empty:
#         continue

#     positives["signal"] = signal
#     positives["method"] = method
#     positives["threshold"] = threshold
#     positives["match"] = np.where(
#         positives["manual_present"] == 1, "TP (model + manual agree)", "FP (model only)"
#     )
#     detected_rows.append(positives)

# detected_df = pd.concat(detected_rows, ignore_index=True)

# # attach message text/actor/timestamp for readability
# detected_df = detected_df.join(
#     level1_df_full[[actor_col, timestamp_col, text_col]], on=turn_id_col
# )
# detected_df["message_preview"] = detected_df[text_col].astype(str).str.slice(0, 100)

# display_cols = [
#     "signal", "method", turn_id_col, actor_col, timestamp_col,
#     "score", "threshold", "manual_present", "match", "message_preview",
# ]
# detected_df = detected_df[display_cols].sort_values(["signal", "method", "score"], ascending=[True, True, False])

# print(f"{len(detected_df)} total (signal, method, turn) positives detected across all methods.")
# print(f"  TP: {(detected_df['match'].str.startswith('TP')).sum()}  |  "
#       f"FP: {(detected_df['match'].str.startswith('FP')).sum()}")

# out_path_detected = vis_dir / f"{VALIDATION_PARTICIPANT}_metrics_positive_cases.csv"
# detected_df.to_csv(out_path_detected, index=False)
# print(f"[ok] {out_path_detected.name}")

# from google.colab import files
# assert out_path_detected.exists(), f"{out_path_detected} wasn't written -- check the .to_csv() line above ran successfully."
# files.download(str(out_path_detected))

# display(detected_df)

In [ ]:
# ============================================================
# All manually-annotated positives, checked against
# what each method actually scored them (the FN-focused mirror
# of Part 6.4's FP-focused view)
# ============================================================

# annotated_rows = []
# for _, row in f1_df.dropna(subset=["threshold"]).iterrows():
#     signal, method, threshold = row["signal"], row["method"], row["threshold"]

#     subset = merged[(merged["signal"] == signal) & (merged["method"] == method)]
#     positives = subset[subset["manual_present"] == 1].copy()
#     if positives.empty:
#         continue

#     positives["signal"] = signal
#     positives["method"] = method
#     positives["threshold"] = threshold
#     positives["match"] = np.where(
#         positives["score"] >= threshold, "TP (model caught it)", "FN (model missed it)"
#     )
#     annotated_rows.append(positives)

# annotated_df = pd.concat(annotated_rows, ignore_index=True)

# # bring in the original 0/1/2 manual_score (not just the collapsed present flag) --
# # lets you see whether missed cases skew toward implicit (1) vs explicit (2)
# annotated_df = annotated_df.merge(
#     manual_long[[turn_id_col, "signal", "manual_score"]],
#     on=[turn_id_col, "signal"], how="left",
# )

# annotated_df = annotated_df.join(
#     level1_df_full[[actor_col, timestamp_col, text_col]], on=turn_id_col
# )
# annotated_df["message_preview"] = annotated_df[text_col].astype(str).str.slice(0, 100)

# # implicit-vs-explicit miss rate computed BEFORE filtering to FN-only, since
# # it needs both TP and FN present to be a meaningful ratio
# miss_rate_by_level = annotated_df.groupby("manual_score")["match"].value_counts(normalize=True).round(2)

# # keep only the misses (FN)
# annotated_df = annotated_df[annotated_df["match"].str.startswith("FN")].copy()

# display_cols = [
#     "signal", "method", turn_id_col, actor_col, timestamp_col,
#     "manual_score", "score", "threshold", "match", "message_preview",
# ]
# annotated_df = annotated_df[display_cols].sort_values(
#     ["signal", "method", "score"], ascending=[True, True, True]
# )

# print(f"{len(annotated_df)} manually-annotated positives missed across all methods (FN only).")

# print("\nMiss rate by original manual_score level (computed pre-filter, for context):")
# print(miss_rate_by_level)

# out_path_annotated = vis_dir / f"{VALIDATION_PARTICIPANT}_manual_positives_vs_metrics.csv"
# annotated_df.to_csv(out_path_annotated, index=False)
# print(f"\n[ok] {out_path_annotated.name}")

# from google.colab import files
# assert out_path_annotated.exists(), f"{out_path_annotated} wasn't written -- check the .to_csv() line above ran successfully."
# files.download(str(out_path_annotated))

# display(annotated_df)

In [ ]:
# =======================================
# Part 6.3: F1 Heatmap -- signal x method
# =======================================

# f1_method_display_names = {**method_display_names, "llm_zeroshot": "Zero-shot LLM"}

# f1_color_scale = [
#     [0.0, "#F7F7F7"], [0.25, "#D9E8F5"], [0.5, "#7FB3D9"], [0.75, "#3E7CB1"], [1.0, "#1B4F72"],
# ]


# def make_f1_heatmap(f1_df, title):
#     pivot = f1_df.pivot(index="signal", columns="method", values="f1")
#     pivot = pivot[[m for m in F1_METHODS if m in pivot.columns]]
#     pivot.columns = [f1_method_display_names.get(m, m) for m in pivot.columns]

#     z = pivot.values
#     text = [[("n/a" if pd.isna(v) else f"{v:.2f}") for v in row] for row in z]

#     fig = go.Figure(data=go.Heatmap(
#         z=z, x=pivot.columns.tolist(), y=pivot.index.tolist(),
#         colorscale=f1_color_scale, zmin=0, zmax=1,
#         text=text, texttemplate="%{text}", textfont={"size": 12},
#         colorbar=dict(title="F1"),
#         hovertemplate="%{y}<br>%{x}<br>F1: %{z:.3f}<extra></extra>",
#     ))
#     fig.update_layout(
#         title=title, height=110 + 55 * len(pivot.index),
#         width=max(700, 130 * len(pivot.columns)),
#         xaxis=dict(side="bottom", tickangle=-30), yaxis=dict(autorange="reversed"),
#         margin=dict(l=220, b=120),
#     )
#     return fig


# fig_f1 = make_f1_heatmap(
#     f1_df, f"{VALIDATION_PARTICIPANT} — F1 vs. Manual Codes by Signal & Method<br>"
#            f"<sup>in-sample best threshold, lexicon excluded</sup>"
# )
# out_path_f1 = vis_dir / f"{VALIDATION_PARTICIPANT}_f1_validation_heatmap.html"
# fig_f1.write_html(str(out_path_f1))
# print(f"[ok] {out_path_f1.name}")
# fig_f1.show()

## [INACTIVE] Part X: Cross-Method Correlation Matrices

In this section, for each active signal, we compute pairwise Spearman rank correlation between every pair of
methods that scored it, using every participant's real, scored turns. Correlations are calculated using turns across all participants for each signal. A pair of methods is only
compared on turns that both actually scored. These correlations represent **agreement between methods**, not actual accuracy.

In [ ]:
# corr_dir = vis_dir  # save heatmaps alongside the other visualizations
# signal_corr_tables = {}  # signal_name -> corr DataFrame, kept for the CSV export below
#
# for construct, sigs in signals_by_construct.items():
#     for signal_name in sigs:
#         sig_methods = active_signals.loc[
#             (active_signals["construct"] == construct) & (active_signals["signal"] == signal_name),
#             "method"
#         ].tolist()
#         if len(sig_methods) < 2: # signal must have >= methods to compute correlations
#             print(f"  [skip] {signal_name}: only {len(sig_methods)} method(s) active, nothing to correlate")
#             continue

#         # Pool every participant's scored turns for this signal into one wide table:
#         # rows = turn_id (unique across participants already), columns = method
#         per_method_series = {}
#         for method_name in sig_methods:
#             frames = []
#             for metrics_path in sorted(metrics_dir.glob(f"*_{slugify(signal_name)}_{method_name}.csv")):
#                 df = pd.read_csv(metrics_path)
#                 score_col = get_score_col(df)
#                 frames.append(df[[turn_id_col, score_col]].rename(columns={score_col: method_name}))
#             if not frames:
#                 continue
#             combined = pd.concat(frames, ignore_index=True)
#             per_method_series[method_name] = combined.set_index(turn_id_col)[method_name]

#         if len(per_method_series) < 2:
#             print(f"  [skip] {signal_name}: metrics files missing for enough methods")
#             continue

#         wide = pd.DataFrame(per_method_series)  # NaN where a method didn't score that turn
#         corr = wide.corr(method="spearman")     # pds handles pairwise-complete-obs so a method pair is only compared on turns that both actually scored
#         signal_corr_tables[signal_name] = corr
#         n_complete = wide.dropna().shape[0]    # count of turns that every method for that signal actually scored

#         methods_here = corr.columns.tolist()
#         fig = go.Figure(data=go.Heatmap(
#             z=corr.values,
#             x=[method_display_names.get(m, m) for m in methods_here],
#             y=[method_display_names.get(m, m) for m in methods_here],
#             zmin=-1, zmax=1,
#             colorscale="RdBu", reversescale=True,
#             text=corr.round(2).values, texttemplate="%{text}",
#             colorbar=dict(title="Spearman ρ"),
#         ))
#         fig.update_layout(
#             title=f"{signal_name} — Cross-Method Spearman Correlation (n={n_complete} turns, all methods)",
#             height=420, width=460,
#         )
#         out_path = vis_dir / f"{slugify(signal_name)}_method_correlation.html"
#         fig.write_html(str(out_path))
#         print(f"[ok] {out_path.name}  (n_complete={n_complete})")
#         fig.show()

# # Flat CSV export: one row per (signal, method_a, method_b, rho)
# export_rows = []
# for signal_name, corr in signal_corr_tables.items():
#     methods_here = corr.columns.tolist()
#     for i, m1 in enumerate(methods_here):
#         for m2 in methods_here[i + 1:]:
#             export_rows.append({"signal": signal_name, "method_a": m1, "method_b": m2, "spearman_rho": corr.loc[m1, m2]})
# corr_export_path = vis_dir / "cross_method_correlations.csv"
# pd.DataFrame(export_rows).to_csv(corr_export_path, index=False)
# print(f"[ok] {corr_export_path.name}")

In [ ]:
# Save correlation heatmaps to box
# from boxsdk.exception import BoxAPIException

# def _upsert_to_box(file_path, folder_id):
#     try:
#         client.folder(folder_id).upload(str(file_path))
#         print(f"  uploaded (new) {file_path.name}")
#     except BoxAPIException as e:
#         if e.code == 'item_name_in_use':
#             existing_id = e.context_info['conflicts']['id']
#             client.file(existing_id).update_contents(str(file_path))
#             print(f"  updated (overwrote) {file_path.name}")
#         else:
#             raise

# # Upload the correlation heatmaps
# for corr_html in vis_dir.glob("*_method_correlation.html"):
#     _upsert_to_box(corr_html, box_visualizations_folder_id)

# # Upload the flat correlation CSV
# _upsert_to_box(corr_export_path, box_visualizations_folder_id)

## [DEPRECATED] Part X: Ground-Truth Evaluation on Synthetic Data

The sections in this part visualize synthetic/generated user-AI turns and trajectories. They evaluate the different scoring methods against hand-labeled ground truth examples.

* X.1 uses the methods to evaluate independent calibration sentences that exhibit the signals at different levels: hard positive, slight positive, edge-cases, and negative controls (no trajectories)
* X.2 uses the methods to evaluate synthetic trajectories for each signal. These user-AI turns are meant to show increasing trajectories over time.
* X.3 computes quantitative correlations (inter-method agreement, trajectory-vs-known-arc, user-vs-AI) on that same synthetic data.


None of these reflects real participant data. These parts are meant to explore how the methods behave when the "correct" answer is already known.





### X.1: Calibration Sentence Comparison

This calibration set consists of stand-alone user-AI turns. Each user turn is an independent, hand-labeled test case (e.g. a positive paraphrase of an ECR-R item, an edge case, or an unrelated negative control). Each AI turn is an actual ChatGPT response to the user input.

**Grouped bar chart represents method scores within a signal**: Each cluster of bars represents a calibration sentence and each bar
within the cluster represents a method.

**Sentences are ordered left to right by ground truth strength of the signal**: positive examples, -> edge cases -> negative
controls. This means that
bars should appear higher on the left (for stronger positves) and low on the right (for negative control) with edge cases in between.

**Scores are normalized (0–1) before plotting**:
Since the methods have different scales (lexicon is a frequency-style count that
can exceed 1; CCR/SBERT are cosine similarities that mainly fall around 0.1–0.7; zero-shot is a
0–1 probability), we normalize to see a more direct comparison. However, if you hover on the bar, you can see the actual score for the method.

*Note: this data is a small, designed set of sentences (illustrative test cases,
not a real representative sample), ao this serves as a proof-of-concept demonstration before testing the methods on real user data.*

In [ ]:
### Part X.1: Calibration Sentence Comparison

# calibration_participant_id = "GT_calibration"  # <-- set to match your level1 file's participant_id

# calib_level1_path = level1_logs_dir / f"{calibration_participant_id}_level1.csv"
# if not calib_level1_path.exists():
#     print(f"[missing] {calib_level1_path.name} — check calibration_participant_id above.")
# else:
#     calib_df = pd.read_csv(calib_level1_path)
#     calib_df = calib_df[calib_df[actor_col] == "user"].copy()
#     category_order = {"positive_example": 0, "edge_case": 1, "negative_control": 2}
#     calib_df["_sort_key"] = calib_df["category"].map(category_order).fillna(1)
#     calib_df = calib_df.sort_values(["target_signal", "_sort_key"]).reset_index(drop=True)

#     # Short signal abbreviations for the outer axis grouping.
#     signal_names = {
#         "Abandonment & Loss Worry": "Abandonment & Loss Worry",
#         "Relational Imbalance": "Relational Imbalance",
#         "Relationship Self-doubt": "Relationship Self-doubt",
#         "Fear of Being Known": "Fear of Being Known",
#         "Anger at Unmet Needs": "Anger at Unmet Needs",
#         "NONE": "control",
#     }

#     def clean_subtype(subtype: str) -> str:
#         # Strip bracketed tags like "[false positive]" / "[positive]" into a
#         # readable prefix rather than dropping them -- that's exactly the info
#         # we want visible per-bar.
#         return str(subtype).replace("_", " ")

#     rows = []
#     for _, row in calib_df.iterrows():
#         entry = {
#             "turn_id": row[turn_id_col],
#             "signal_abbrev": signal_names.get(row["target_signal"], row["target_signal"]),
#             "subtype": row["subtype"],              # <-- add this line
#             "raw_text": str(row[text_col]),          # <-- add this line
#             "label": f"{clean_subtype(row['subtype'])}<br><i>{str(row[text_col])[:40]}…</i>",
#             "category": row["category"],
#         }
#         if row["target_signal"] == "NONE":
#             target_sigs = list({s for sigs in signals_by_construct.values() for s in sigs})
#         else:
#             target_sigs = [row["target_signal"]]
#         for method in methods_order:
#             vals = []
#             for sig_name in target_sigs:
#                 metrics_path = metrics_dir / f"{calibration_participant_id}_{slugify(sig_name)}_{method}.csv"
#                 if not metrics_path.exists():
#                     continue
#                 df = pd.read_csv(metrics_path)
#                 score_col = get_score_col(df)
#                 v = df.loc[df[turn_id_col] == row[turn_id_col], score_col]
#                 if len(v):
#                     vals.append(v.values[0])
#             entry[method] = sum(vals) / len(vals) if vals else None
#         rows.append(entry)

#     calib_scores = pd.DataFrame(rows)

#     calib_scores_norm = calib_scores.copy()
#     for method in methods_order:
#         col = calib_scores[method].astype(float)
#         rng = col.max() - col.min()
#         calib_scores_norm[method] = (col - col.min()) / rng if rng > 0 else col * 0

#     method_colors = {"ccr": "#4C72B0", "lexicon": "#DD8452", "sbert": "#55A868", "llm_zeroshot": "#C44E52"}

#     # Multi-level x-axis: [outer=signal, inner=sentence label] -- Plotly groups and
#     # visually separates by the outer level automatically when x is given as a
#     # list of two equal-length arrays.
#     x_levels = [calib_scores_norm["signal_abbrev"].tolist(), calib_scores_norm["label"].tolist()]

#     fig = go.Figure()
#     for method in methods_order:
#         fig.add_trace(go.Bar(
#             x=x_levels, y=calib_scores_norm[method],
#             name=method_display_names.get(method, method),
#             marker_color=method_colors.get(method, "#888"),
#             hovertext=[f"raw score: {v:.3f}" for v in calib_scores[method]],
#             hoverinfo="text+name",
#         ))

#     fig.update_layout(
#         barmode="group",
#         title=f"{calibration_participant_id} — Method Scores on Calibration Sentences (normalized 0-1)",
#         xaxis_title=None,
#         yaxis_title="normalized score",
#         height=600, width=max(1100, 110 * len(calib_scores_norm)),
#         legend=dict(orientation="h", y=1.06, x=0.5, xanchor="center"),
#         margin=dict(b=140),
#     )
#     out_path = vis_dir / f"{calibration_participant_id}_method_comparison.html"
#     fig.write_html(str(out_path))
#     print(f"[ok] {out_path.name}")
#     fig.show()

In [ ]:
# ### Part X.1b: Curated Subset (poster-sized chart)

# # Narrative ordering: paraphrased -> slight/implicit positive -> false positive -> control
# SUBTYPE_BUCKET = {
#     "paraphrased_item_7": ("1_paraphrased", "Strong Positive", 0),
#     "[slight positive] mixed anger hurt": ("2_slight", "Slight/Implicit Positive", 0),
#     "[slight positive] uncertain self-doubt": ("2_slight", "Slight/Implicit Positive", 1),
#     "[positive] sarcasm": ("2_slight", "Slight/Implicit Positive", 2),  # <-- pushed to the right
#     "[false positive] reversed imbalance": ("3_false_positive", "False Positive", 0),
#     "[false positive] excessive_self_confidence": ("3_false_positive", "False Positive", 0),
#     "recipe": ("4_control", "Control", 0),
#     "negative_emotions": ("4_control", "Control", 1),
# }
# curated_calib_df = calib_df[calib_df["subtype"].isin(SUBTYPE_BUCKET)].copy()
# curated_calib_df["_bucket_key"] = curated_calib_df["subtype"].map(lambda s: SUBTYPE_BUCKET[s][0])
# curated_calib_df["_order_key"] = curated_calib_df["subtype"].map(lambda s: SUBTYPE_BUCKET[s][2])
# curated_calib_df = curated_calib_df.sort_values(["_bucket_key", "_order_key"]).reset_index(drop=True)


# SHORT_SUBTYPE_LABEL = {
#     "paraphrased_item_7": "Paraphrased ECR-R",
#     "[slight positive] mixed anger hurt": "Mixed Anger/Hurt",
#     "[slight positive] uncertain self-doubt": "Uncertain Self-Doubt",
#     "[positive] sarcasm": "Sarcasm",
#     "[false positive] reversed imbalance": "Reversed Imbalance",
#     "[false positive] excessive_self_confidence": "Excessive Confidence",
#     "recipe": "Recipe",
#     "negative_emotions": "Negative Emotions",
# }

# def wrap_label(subtype, sentence):
#     # Short custom tag instead of the raw subtype string -- much shorter,
#     # avoids the bucket-boundary collision. Full sentence stays in hover only.
#     return SHORT_SUBTYPE_LABEL.get(subtype, clean_subtype(subtype))


# curated_rows = []
# for _, row in curated_calib_df.iterrows():
#     bucket_key, bucket_display, _order = SUBTYPE_BUCKET[row["subtype"]]
#     sig_short = signal_names.get(row["target_signal"], row["target_signal"])
#     entry = {
#         "turn_id": row[turn_id_col],
#         "bucket": bucket_display,
#         "signal_abbrev": sig_short,
#         "subtype": row["subtype"],
#         "raw_text": str(row[text_col]),
#         "wrapped_label": wrap_label(row["subtype"], str(row[text_col])),
#         "category": row["category"],
#     }
#     if row["target_signal"] == "NONE":
#         target_sigs = list({s for sigs in signals_by_construct.values() for s in sigs})
#     else:
#         target_sigs = [row["target_signal"]]
#     for method in methods_order:
#         vals = []
#         for sig_name in target_sigs:
#             metrics_path = metrics_dir / f"{calibration_participant_id}_{slugify(sig_name)}_{method}.csv"
#             if not metrics_path.exists():
#                 continue
#             df = pd.read_csv(metrics_path)
#             score_col = get_score_col(df)
#             v = df.loc[df[turn_id_col] == row[turn_id_col], score_col]
#             if len(v):
#                 vals.append(v.values[0])
#         entry[method] = sum(vals) / len(vals) if vals else None
#     curated_rows.append(entry)

# curated_calib_scores = pd.DataFrame(curated_rows)

# curated_calib_scores_norm = curated_calib_scores.copy()
# for method in methods_order:
#     col = curated_calib_scores[method].astype(float)
#     rng = col.max() - col.min()
#     curated_calib_scores_norm[method] = (col - col.min()) / rng if rng > 0 else col * 0

# x_levels = [
#     curated_calib_scores_norm["bucket"].tolist(),
#     curated_calib_scores_norm["wrapped_label"].tolist(),
# ]

# curated_fig = go.Figure()
# for method in methods_order:
#     curated_fig.add_trace(go.Bar(
#         x=x_levels, y=curated_calib_scores_norm[method],
#         name=method_display_names.get(method, method),
#         marker_color=method_colors.get(method, "#888"),
#         hovertext=[f"raw score: {v:.3f}" for v in curated_calib_scores[method]],
#         hoverinfo="text+name",
#     ))

# curated_fig.update_layout(
#     barmode="group",
#     title=f"{calibration_participant_id} — Curated Calibration Sentences (normalized 0-1)",
#     xaxis_title=None,
#     yaxis=dict(title="normalized score", title_font=dict(size=16), tickfont=dict(size=14)),
#     xaxis=dict(tickangle=0, tickfont=dict(size=13), automargin=True),
#     height=700, width=max(1600, 230 * len(curated_calib_scores_norm)),
#     legend=dict(orientation="h", y=1.06, x=0.5, xanchor="center", font=dict(size=15)),
#     margin=dict(b=160, t=100, l=80, r=40),
#     font=dict(size=16),
#     title_font=dict(size=20),
#     bargap=0.15,
#     bargroupgap=0.05,
# )
# curated_fig.show()

# out_html_path = vis_dir / f"{calibration_participant_id}_method_comparison_curated.html"
# curated_fig.write_html(str(out_html_path))
# print(f"[ok] {out_html_path.name}")

# png_path = vis_dir / f"{calibration_participant_id}_method_comparison_curated.png"
# try:
#     curated_fig.write_image(str(png_path), width=curated_fig.layout.width,
#                              height=curated_fig.layout.height, scale=3)
#     print(f"[ok] {png_path.name}")
# except Exception as e:
#     print(f"[FAILED] PNG export: {type(e).__name__}: {e}")
#     print("  -> Check kaleido==0.2.1 is installed and the runtime was restarted after installing it.")

### Part X.2: Synthetic Trajectory Validation

Each of the five `GT_traj_*` files contains a synthetic conversation spanning six weeks (~10 user turns per week, along with corresponding AI turns). During data generation, the proportion of user messages expressing the target signal was intentionally increased over time following a predefined trajectory:

0% => 10% => 30% => 50% => 70% => 90%

For each week, detector outputs were aggregated into a weekly mean score. For better comparison across methods, scores were min–max normalized within each signal-method combination. Weekly user and AI scores are shown separately.

Each panel below corresponds to one signal category. Colored lines show the weekly mean detector scores for each method, while the dashed black line represents the intended trajectory specified during synthetic data generation. Methods that successfully capture the increasing prevalence of a signal should exhibit a trajectory that roughly follows the shape of the reference line.

*Note: These synthetic conversations are intentionally small-scale and designed as proof-of-concept examples. Our goal is not to evaluate real-world estimation but rather to see whether different detection methods can identify an obvious, known trajectory pattern under controlled conditions.*

In [ ]:
# trajectory_signals = {
#     "ALW": "Abandonment & Loss Worry",
#     "RI": "Relational Imbalance",
#     "RSD": "Relationship Self-doubt",
#     "FOBK": "Fear of Being Known",
#     "AUN": "Anger at Unmet Needs",
# }

# method_colors = {
#     "ccr": "#4C72B0",
#     "lexicon": "#DD8452",
#     "sbert": "#55A868",
#     "llm_zeroshot": "#C44E52",
# }

# def normalize_series(s: pd.Series) -> pd.Series:
#     rng = s.max() - s.min()
#     return (s - s.min()) / rng if rng > 0 else s * 0

# def weekly_mean_for_actor(df, score_col, week_by_turn, all_weeks):
#     sub = df[df[turn_id_col].isin(week_by_turn)].copy()
#     sub["week"] = sub[turn_id_col].map(week_by_turn)
#     return sub.groupby("week")[score_col].mean().reindex(all_weeks)

# fig = make_subplots(
#     rows=len(trajectory_signals),
#     cols=1,
#     subplot_titles=list(trajectory_signals.values()),
#     vertical_spacing=0.08,
# )

# for r, (abbrev, signal_name) in enumerate(
#     trajectory_signals.items(), start=1
# ):

#     level1_path = level1_logs_dir / f"GT_{abbrev}_trajectory_level1.csv"

#     if not level1_path.exists():
#         print(f"[missing] {level1_path.name} — skipping {signal_name}")
#         continue

#     level1_df = pd.read_csv(level1_path)

#     participant_id = (
#         level1_df[participant_col].iloc[0]
#         if participant_col in level1_df.columns
#         else f"GT_traj_{abbrev}"
#     )

#     all_weeks = sorted(level1_df["week"].dropna().unique())

#     user_rows = level1_df[level1_df["actor"] == "user"]
#     ai_rows = level1_df[level1_df["actor"] == "AI"]

#     week_by_turn_user = (
#         user_rows.set_index(turn_id_col)["week"].to_dict()
#     )
#     week_by_turn_ai = (
#         ai_rows.set_index(turn_id_col)["week"].to_dict()
#     )

#     intensity_by_week = (
#         user_rows
#         .dropna(subset=["intended_intensity"])
#         .groupby("week")["intended_intensity"]
#         .first()
#         .reindex(all_weeks)
#         .fillna(0)
#     )

#     for method in methods_order:

#         metrics_path = (
#             metrics_dir
#             / f"{participant_id}_{slugify(signal_name)}_{method}.csv"
#         )

#         if not metrics_path.exists():
#             print(f"  [missing] {metrics_path.name}")
#             continue

#         df = pd.read_csv(metrics_path)
#         score_col = get_score_col(df)

#         raw_user = weekly_mean_for_actor(
#             df, score_col, week_by_turn_user, all_weeks
#         )
#         raw_ai = weekly_mean_for_actor(
#             df, score_col, week_by_turn_ai, all_weeks
#         )

#         weekly_user = normalize_series(raw_user)
#         weekly_ai = normalize_series(raw_ai)

#         method_label = method_display_names.get(method, method)

#         # User trace (legend only on first subplot)
#         fig.add_trace(
#             go.Scatter(
#                 x=weekly_user.index,
#                 y=weekly_user.values,
#                 mode="lines+markers",
#                 name=f"{method_label} (user)",
#                 legendgroup=f"{method}_user",
#                 showlegend=(r == 1),
#                 line=dict(
#                     color=method_colors.get(method, "#888"),
#                     width=2,
#                     dash="solid",
#                 ),
#                 hovertext=[
#                     f"raw score: {v:.3f}"
#                     for v in raw_user.values
#                 ],
#                 hoverinfo="text+name+x",
#             ),
#             row=r,
#             col=1,
#         )

#         # AI trace (legend only on first subplot)
#         fig.add_trace(
#             go.Scatter(
#                 x=weekly_ai.index,
#                 y=weekly_ai.values,
#                 mode="lines+markers",
#                 name=f"{method_label} (AI)",
#                 legendgroup=f"{method}_ai",
#                 showlegend=(r == 1),
#                 line=dict(
#                     color=method_colors.get(method, "#888"),
#                     width=1.5,
#                     dash="dot",
#                 ),
#                 marker=dict(
#                     symbol="diamond-open",
#                     size=6,
#                 ),
#                 hovertext=[
#                     f"raw score: {v:.3f}"
#                     for v in raw_ai.values
#                 ],
#                 hoverinfo="text+name+x",
#             ),
#             row=r,
#             col=1,
#         )

#     intensity_norm = normalize_series(intensity_by_week)

#     fig.add_trace(
#         go.Scatter(
#             x=intensity_norm.index,
#             y=intensity_norm.values,
#             mode="lines",
#             name="intended intensity",
#             legendgroup="intended",
#             showlegend=(r == 1),
#             line=dict(
#                 color="black",
#                 width=1.5,
#                 dash="dash",
#             ),
#             hovertext=[
#                 f"raw week-level intensity: {v:.0f}"
#                 for v in intensity_by_week.values
#             ],
#             hoverinfo="text+name+x",
#         ),
#         row=r,
#         col=1,
#     )

#     fig.update_yaxes(
#         title_text="normalized score",
#         range=[-0.05, 1.05],
#         row=r,
#         col=1,
#     )
#
#     if r == len(trajectory_signals):
#         fig.update_xaxes(
#             title_text="week",
#             row=r,
#             col=1,
#         )

# fig.update_layout(
#     title=(
#         "Synthetic Trajectory Validation — User vs. AI Turns<br>"
#         "<sup>solid = user turns, dotted = AI turns, "
#         "both normalized 0–1 per signal</sup>"
#     ),
#     height=340 * len(trajectory_signals),
#     width=850,
#     legend=dict(
#         orientation="h",
#         y=-0.05,
#         x=0.5,
#         xanchor="center",
#         font=dict(size=10),
#     ),
#     margin=dict(t=90),
# )

# out_path = vis_dir / "trajectory_validation_user_vs_ai.html"

# fig.write_html(str(out_path))
# print(f"[ok] {out_path.name}")

# fig.show()

### Part X.3: Quantitative Correlations

Two correlation checks, both using Spearman rank correlation (not affected by method scoring scales):

1. **Inter-method Agreement**: for the five synthetic trajectory files, do the methods rank the turns in the same order? High correlation suggests that the methods agree on their scores for a signal. Low correlation suggests the methods are picking up on different things and hence have more varied scores.
2. **Trajectory Correlation**: for the five synthetic trajectory files, does each method's weekly-aggregated score correlate with the known, designed intensity arc? (computed for user AND AI trajectories)
3. **User-AI Trajectory Correlation**: for the five synthetic trajectory files, does the user's signal trajectory correlate with the AI signal trajectory?

*Note: trajectory correlations use only 6 weekly points per signal. So these are more descriptive/illustrative findings, not statistically significant results.*

In [ ]:
# ### Part X.3: Quantitative Correlations

# from scipy.stats import spearmanr

# # Explicit method list -- matches signals_config.csv's distinct "method" column values.
# # Defined here directly rather than assumed, since it wasn't actually defined elsewhere.
# methods_order = ["ccr", "lexicon", "sbert", "llm_zeroshot"]

# # ---------- 1. Inter-method correlation, per participant + signal ----------

# from scipy.stats import spearmanr

# methods_order = ["ccr", "lexicon", "sbert", "llm_zeroshot"]

# trajectory_signals = {
#     "ALW": "Abandonment & Loss Worry",
#     "RI": "Relational Imbalance",
#     "RSD": "Relationship Self-doubt",
#     "FOBK": "Fear of Being Known",
#     "AUN": "Anger at Unmet Needs",
# }


# def compute_method_correlations(participant_id, signal_name,
#                                 methods=methods_order):
#     """
#     Compute pairwise agreement between scoring methods separately for each
#     synthetic trajectory participant (GT_traj_ALW, GT_traj_RI, GT_traj_RSD,
#     GT_traj_FOBK, GT_traj_AUN).
#     """
#     score_series = {}

#     for method in methods:
#         path = metrics_dir / f"{participant_id}_{slugify(signal_name)}_{method}.csv"

#         if not path.exists():
#             continue

#         df = pd.read_csv(path)
#         score_col = get_score_col(df)
#         # correlations computed at turn-level
#         score_series[method] = df.set_index(turn_id_col)[score_col]

#     results = []

#     method_list = list(score_series.keys())

#     for i, m1 in enumerate(method_list):
#         for m2 in method_list[i + 1:]:

#             merged = pd.concat(
#                 [score_series[m1], score_series[m2]],
#                 axis=1,
#                 keys=[m1, m2]
#             ).dropna()

#             if (
#                 len(merged) < 3
#                 or merged[m1].nunique() <= 1
#                 or merged[m2].nunique() <= 1
#             ):
#                 rho, p = float("nan"), float("nan")
#             else:
#                 rho, p = spearmanr(merged[m1], merged[m2])

#             results.append({
#                 "participant_id": participant_id,
#                 "signal": signal_name,
#                 "method_a": m1,
#                 "method_b": m2,
#                 "rho": rho,
#                 "p": p,
#                 "n": len(merged),
#             })

#     return pd.DataFrame(results)

# # Compute correlations for ALL synthetic trajectories

# all_corrs = []

# for abbrev, signal_name in trajectory_signals.items():

#     participant_id = f"GT_traj_{abbrev}"

#     all_corrs.append(
#         compute_method_correlations(
#             participant_id,
#             signal_name
#         )
#     )

# method_corr_df = pd.concat(all_corrs, ignore_index=True)

# method_corr_df.to_csv(
#     vis_dir / "inter_method_correlations.csv",
#     index=False,
# )

# # ---------- 2. Correlation with the known trajectory arc ----------

# def compute_trajectory_correlations(actor_filter: str) -> pd.DataFrame:
#     """
#     Correlates each method's weekly-aggregated score (restricted to the given actor:
#     'user' or 'AI') against the known intended_intensity arc, per signal.
#     """
#     results = []
#     for abbrev, signal_name in trajectory_signals.items():
#         level1_path = level1_logs_dir / f"GT_{abbrev}_trajectory_level1.csv"
#         if not level1_path.exists():
#             print(f"[missing] {level1_path.name}")
#             continue
#         level1_df = pd.read_csv(level1_path)
#         participant_id = level1_df[participant_col].iloc[0] if participant_col in level1_df.columns \
#             else f"GT_traj_{abbrev}"

#         all_weeks = sorted(level1_df["week"].dropna().unique())
#         actor_rows = level1_df[level1_df["actor"] == actor_filter]
#         week_by_turn = actor_rows.set_index(turn_id_col)["week"].to_dict()

#         # intended_intensity is defined on user rows only (it's a property of the
#         # user's message composition that week) -- always pull it from user rows,
#         # regardless of which actor's scores we're correlating against it.
#         user_rows = level1_df[level1_df["actor"] == "user"]
#         intensity_by_week = (
#             user_rows.dropna(subset=["intended_intensity"])
#             .groupby("week")["intended_intensity"].first()
#             .reindex(all_weeks).fillna(0)
#         )

#         for method in methods_order:
#             metrics_path = metrics_dir / f"{participant_id}_{slugify(signal_name)}_{method}.csv"
#             if not metrics_path.exists():
#                 continue
#             df = pd.read_csv(metrics_path)
#             score_col = get_score_col(df)
#             df = df[df[turn_id_col].isin(week_by_turn)].copy()
#             df["week"] = df[turn_id_col].map(week_by_turn)
#             weekly_mean = df.groupby("week")[score_col].mean().reindex(all_weeks)

#             common_weeks = weekly_mean.dropna().index.intersection(intensity_by_week.index)
#             if len(common_weeks) < 3 or weekly_mean.loc[common_weeks].nunique() <= 1:
#                 rho, p = float("nan"), float("nan")
#             else:
#                 rho, p = spearmanr(weekly_mean.loc[common_weeks], intensity_by_week.loc[common_weeks])
#             results.append({
#                 "signal": signal_name, "method": method, "actor": actor_filter,
#                 "rho": rho, "p": p, "n_weeks": len(common_weeks),
#             })
#     return pd.DataFrame(results)


# # ---------- 3. User vs. AI score correlation ----------

# def compute_user_ai_correlation() -> pd.DataFrame:
#     """
#     Correlates each method's weekly-aggregated USER score against its weekly-aggregated
#     AI score, per signal -- a direct test of whether AI-turn scores are tracking the
#     user's own trajectory (leakage) rather than moving independently.
#     """
#     results = []
#     for abbrev, signal_name in trajectory_signals.items():
#         level1_path = level1_logs_dir / f"GT_{abbrev}_trajectory_level1.csv"
#         if not level1_path.exists():
#             print(f"[missing] {level1_path.name}")
#             continue
#         level1_df = pd.read_csv(level1_path)
#         participant_id = level1_df[participant_col].iloc[0] if participant_col in level1_df.columns \
#             else f"GT_traj_{abbrev}"

#         all_weeks = sorted(level1_df["week"].dropna().unique())
#         week_by_turn_user = level1_df[level1_df["actor"] == "user"].set_index(turn_id_col)["week"].to_dict()
#         week_by_turn_ai = level1_df[level1_df["actor"] == "AI"].set_index(turn_id_col)["week"].to_dict()

#         for method in methods_order:
#             metrics_path = metrics_dir / f"{participant_id}_{slugify(signal_name)}_{method}.csv"
#             if not metrics_path.exists():
#                 continue
#             df = pd.read_csv(metrics_path)
#             score_col = get_score_col(df)

#             user_df = df[df[turn_id_col].isin(week_by_turn_user)].copy()
#             user_df["week"] = user_df[turn_id_col].map(week_by_turn_user)
#             weekly_user = user_df.groupby("week")[score_col].mean().reindex(all_weeks)

#             ai_df = df[df[turn_id_col].isin(week_by_turn_ai)].copy()
#             ai_df["week"] = ai_df[turn_id_col].map(week_by_turn_ai)
#             weekly_ai = ai_df.groupby("week")[score_col].mean().reindex(all_weeks)

#             common_weeks = weekly_user.dropna().index.intersection(weekly_ai.dropna().index)
#             if len(common_weeks) < 3 or weekly_user.loc[common_weeks].nunique() <= 1 \
#                     or weekly_ai.loc[common_weeks].nunique() <= 1:
#                 rho, p = float("nan"), float("nan")
#             else:
#                 rho, p = spearmanr(weekly_user.loc[common_weeks], weekly_ai.loc[common_weeks])
#             results.append({
#                 "signal": signal_name, "method": method, "rho": rho, "p": p, "n_weeks": len(common_weeks),
#             })
#     return pd.DataFrame(results)


# # ---------- Run everything and save ----------

# user_trajectory_corr_df = compute_trajectory_correlations("user")
# ai_trajectory_corr_df = compute_trajectory_correlations("AI")
# user_ai_corr_df = compute_user_ai_correlation()


# user_trajectory_corr_df.to_csv(vis_dir / "trajectory_intensity_correlations_user.csv", index=False)
# ai_trajectory_corr_df.to_csv(vis_dir / "trajectory_intensity_correlations_ai.csv", index=False)
# user_ai_corr_df.to_csv(vis_dir / "trajectory_intensity_correlations_user-ai.csv", index=False)  # <-- fixed name

# # Combined side-by-side table with the gap column, used by Part 4.4's Heatmap 4
# trajectory_corr_df = pd.concat([user_trajectory_corr_df, ai_trajectory_corr_df], ignore_index=True)
# trajectory_corr_wide = trajectory_corr_df.pivot_table(
#     index=["signal", "method"], columns="actor", values="rho"
# ).reset_index()
# trajectory_corr_wide["user_minus_ai_gap"] = trajectory_corr_wide["user"] - trajectory_corr_wide["AI"]
# trajectory_corr_wide.to_csv(vis_dir / "trajectory_intensity_correlations_user_vs_ai.csv", index=False)




In [ ]:
# color_pos_neg_scale = [
#     [0.0, "#2166AC"], [0.25, "#67A9CF"], [0.5, "#F7F7F7"], [0.75, "#EF8A62"], [1.0, "#B2182B"],
# ]

# def make_correlation_heatmap(pivot_df: pd.DataFrame, title: str, zmin=-1, zmax=1,
#                               colorbar_title="Spearman<br>rho"):
#     z = pivot_df.values
#     text = [[("n/a" if pd.isna(v) else f"{v:.2f}") for v in row] for row in z]
#     fig = go.Figure(data=go.Heatmap(
#         z=z, x=pivot_df.columns.tolist(), y=pivot_df.index.tolist(),
#         colorscale=color_pos_neg_scale, zmin=zmin, zmax=zmax, zmid=0,
#         text=text, texttemplate="%{text}", textfont={"size": 12},
#         colorbar=dict(title=colorbar_title),
#         hovertemplate="%{y}<br>%{x}<br>value: %{z:.3f}<extra></extra>",
#     ))
#     fig.update_layout(
#         title=title, height=110 + 55 * len(pivot_df.index),
#         width=max(700, 130 * len(pivot_df.columns)),
#         xaxis=dict(side="bottom", tickangle=-30), yaxis=dict(autorange="reversed"),
#         margin=dict(l=180, b=120),
#     )
#     return fig


# # ---------- Heatmap 1: inter-method correlations (unchanged from before) ----------

# method_corr_df["pair"] = method_corr_df["method_a"] + " vs " + method_corr_df["method_b"]
# method_pivot = method_corr_df.pivot(index="signal", columns="pair", values="rho")

# fig1 = make_correlation_heatmap(
#     method_pivot, f"Inter-Method Correlation (Synthetic Trajectories)"
# )
# out_path1 = vis_dir / "synthetic_trajectories_method_correlation_heatmap.html"
# fig1.write_html(str(out_path1))
# print(f"[ok] {out_path1.name}")
# fig1.show()


# # ---------- Heatmap 2: trajectory correlation -- USER turns ----------

# user_pivot = user_trajectory_corr_df.pivot(index="signal", columns="method", values="rho")
# user_pivot = user_pivot[[m for m in methods_order if m in user_pivot.columns]]
# user_pivot.columns = [method_display_names.get(m, m) for m in user_pivot.columns]

# fig2 = make_correlation_heatmap(
#     user_pivot, "Trajectory Correlation with Known Intensity Arc -- User Turns"
# )
# out_path2 = vis_dir / "trajectory_intensity_correlation_heatmap_user.html"
# fig2.write_html(str(out_path2))
# print(f"[ok] {out_path2.name}")
# fig2.show()


# # ---------- Heatmap 3: trajectory correlation -- AI turns ----------

# ai_pivot = ai_trajectory_corr_df.pivot(index="signal", columns="method", values="rho")
# ai_pivot = ai_pivot[[m for m in methods_order if m in ai_pivot.columns]]
# ai_pivot.columns = [method_display_names.get(m, m) for m in ai_pivot.columns]

# fig3 = make_correlation_heatmap(
#     ai_pivot, "Trajectory Correlation with Known Intensity Arc -- AI Turns"
# )
# out_path3 = vis_dir / "trajectory_intensity_correlation_heatmap_ai.html"
# fig3.write_html(str(out_path3))
# print(f"[ok] {out_path3.name}")
# fig3.show()

# # ---------- Heatmap 4: USer-AI trajectory correlation ----------


# # Heatmap
# user_ai_pivot = user_ai_corr_df.pivot(index="signal", columns="method", values="rho")
# user_ai_pivot = user_ai_pivot[[m for m in methods_order if m in user_ai_pivot.columns]]
# user_ai_pivot.columns = [method_display_names.get(m, m) for m in user_ai_pivot.columns]

# fig5 = make_correlation_heatmap(
#     user_ai_pivot,
#     "User Score vs. AI Score Correlation<br><sup>high rho = AI-turn score tracks user-turn score (possible leakage)</sup>",
# )
# out_path5 = vis_dir / "trajectory_user_ai_score_correlation_heatmap.html"
# fig5.write_html(str(out_path5))
# print(f"[ok] {out_path5.name}")
# fig5.show()
